# Convert to GGUF, then check it

One pass: fetch the checkpoint, work out what architecture it is, convert, quantize, and
verify the result against the original before anything gets published. The last check
loads every file with ComfyUI and the GGUF node exactly the way `Unet Loader (GGUF)` does,
so a file that would render noise is caught here instead of in your workflow.

Fill in 1a to 1e, then Runtime → Run all.

**What changed in this version, and why each one mattered**

* **Norm scales are not quantization scales.** Krea 2 (and other RMSNorm models) name their
  norm weights `*.scale`. The old suffix rule called a clean bf16 checkpoint "already
  quantized" and refused it. A scale now only counts when it pairs with an 8-bit-or-less
  weight, or is ComfyUI quantization metadata.
* **ComfyUI-quantized checkpoints are decoded by ComfyUI.** Files carrying `comfy_quant`
  metadata (fp8, fp8 stored as uint8, int8, **int8 + convrot**, mxfp8, nvfp4, w4a8 …) are
  dequantized through ComfyUI's own code. The old `weight × scale` guess got int8-convrot
  backwards in a way no magnitude check can see - the rotated weights have the right spread
  and the wrong values (per-weight cosine 0.06), which renders as grey/black-and-white noise.
* **Weights the source kept in F32 stay F32.** Krea 2 ships bf16 with its input/output,
  timestep and text-projection layers in fp32. Those are kept F32 in every file, and every
  F32 tensor in the base file is put back if llama-quantize quantized it anyway.
* **Nothing the node can't decode is written.** BF16 or quantized tensors that no GGUF layer
  dequantizes (raw parameters, Conv1d/Conv3d) are stored as F32; Python quantization is
  limited to the types the GGUF node actually implements.
* **Architecture name is chosen per loader.** Upstream city96 has no Krea 2 support;
  `molbal/ComfyUI-GGUF` has a template and detection, `RealRebelAI/ComfyUI-GGUF_KREA-2`
  only whitelists the name. The notebook now picks a name the loader in 1d really accepts.
* **Protect patterns** support the `^prefix` and `*` wildcard syntax forks use.
* **LLMs** convert with a current llama.cpp; the 2024 patched build is only for image models.
* **Final ComfyUI check:** missing keys, every loaded weight compared against the checkpoint
  through the node's own dequantizer, and a forward pass of each quant against the base file.

**Single-row parameters.** A raw `nn.Parameter` stored as `[1, dim]` is one row, and nothing
dequantizes a bare Parameter at load time, so it has to stay F32. `llama-quantize` writes
each tensor's dimension count as `ggml_n_dims()`, which drops trailing 1s, so those
dimensions are compared against the base file and put back.

In [ ]:
# @title 1a · What you're doing { display-mode: "form" }

# @markdown Convert, check, or both. `check only` skips straight to verifying whatever is
# @markdown already in the output folder.
JOB = "convert and check"  # @param ["convert and check", "convert only", "check only"]

# @markdown What kind of model this is. Decides which converter runs and which checks apply.
TASK = "image generation"  # @param ["auto-detect", "text / chat LLM", "vision (VQA, captioning, OCR)", "audio-in LLM", "embedding / reranker", "image generation", "video generation"]

# @markdown ---
# @markdown Scratch space for the toolchain, and where the finished files go.
WORK_DIR = "/content/gguf-work"  # @param {type:"string"}
OUT_DIR = "/content/gguf-out"  # @param {type:"string"}

# @markdown Where the result has to load.
RUNTIME = "auto"  # @param ["auto", "llama.cpp / Ollama", "ComfyUI (GGUF nodes)"]

In [ ]:
# @title 1b · The model { display-mode: "form" }

# @markdown Where the checkpoint comes from.
SRC_FROM = "download link"  # @param ["download link", "local path", "Hugging Face repo"]

# @markdown **download link** — the direct URL. Hugging Face is the Download button on the
# @markdown file page, or `https://huggingface.co/<repo>/resolve/main/<file>`. Civitai looks
# @markdown like `https://civitai.com/api/download/models/<id>?fileId=<id>`.
# @markdown Several URLs, comma separated, are fine - extras are treated as companion files.
SRC_URL = ""  # @param {type:"string"}

# @markdown **local path** — a `.safetensors` file, or a model folder for the LLM tasks.
SRC_PATH = ""  # @param {type:"string"}

# @markdown **Hugging Face repo** — for text/vision/audio/embedding, which need the folder.
# @markdown For an image or video model in diffusers layout (`model_index.json` +
# @markdown `transformer/`), only the transformer folder is downloaded and converted. Prefer the
# @markdown single-file ComfyUI repack (e.g. `Comfy-Org/...`) when one exists - its key names are
# @markdown the ones ComfyUI expects.
SRC_REPO = ""  # @param {type:"string"}
SRC_FILE = ""  # @param {type:"string"}
SRC_REVISION = "main"  # @param {type:"string"}

# @markdown ---
# @markdown Leave the tokens blank if you've set Colab secrets called `HF_TOKEN` and
# @markdown `CIVITAI_TOKEN` (the key icon on the left). Civitai needs one for most downloads.
HF_TOKEN = ""  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}
# @markdown Parallel connections per download.
CONNECTIONS = 16  # @param {type:"slider", min:1, max:16, step:1}

In [ ]:
# @title 1c · What to build { display-mode: "form" }

# @markdown `mix` keeps whatever precision the checkpoint already uses, which is the safest
# @markdown option and the one everything else is quantized from.
WANT_MIX = True  # @param {type:"boolean"}
WANT_8BIT = True  # @param {type:"boolean"}
WANT_4BIT = True  # @param {type:"boolean"}

RECIPE_8BIT = "Q8_0"  # @param ["Q8_0"]
# @markdown K-quants (Q4_K_M …) need llama-quantize to know the architecture. For one it
# @markdown doesn't (Krea 2 and other new families) see `IF_ARCH_UNKNOWN` in 1d.
RECIPE_4BIT = "Q4_K_M"  # @param ["Q4_K_M", "Q4_K_S", "Q4_1", "Q4_0"]
# @markdown Anything else, comma separated. `none` to skip.
EXTRA_QUANTS = "none"  # @param ["none", "Q6_K", "Q5_K_M", "Q5_K_S", "Q3_K_M", "Q2_K"] {allow-input: true}

# @markdown ---
# @markdown If a target can't be built the way you asked, `substitute` builds the nearest
# @markdown thing that can be (Q4_K_M becomes Q4_1, Q6_K becomes Q8_0, and so on) rather
# @markdown than leaving you with nothing.
IF_QUANT_UNAVAILABLE = "substitute the nearest"  # @param ["substitute the nearest", "drop it"]

# @markdown ---
# @markdown For multimodal LLMs: also export the `mmproj` projector, which is the image or
# @markdown audio half of the model. `auto` does it for the vision and audio tasks.
EXPECT_MMPROJ = "auto"  # @param ["auto", "yes", "no"]

# @markdown ---
# @markdown What to do about a file that's already in the output folder from an earlier
# @markdown run. `skip` leaves it where it is and only builds what's missing, which is what
# @markdown you want when a run died partway through or you're adding one more quant to a
# @markdown set. Skipped files are still checked. `rebuild` converts everything again and
# @markdown overwrites.
IF_OUTPUT_EXISTS = "skip"  # @param ["skip", "rebuild"]

In [ ]:
# @title 1d · Architecture { display-mode: "form" }

# @markdown `detect` reads the family off the checkpoint's own tensor names. If nothing
# @markdown matches, the file is still converted - it is not forced into an unrelated family.
# @markdown `force` writes the name below no matter what the weights say.
ARCH_MODE = "detect"  # @param ["detect", "force"]
ARCH_NAME = ""  # @param {type:"string"}

# @markdown ---
# @markdown Already-quantized checkpoints (fp8/int8/fp4 weights next to their scales).
# @markdown `unpack` rebuilds float weights - through ComfyUI's own dequantizer when the file
# @markdown carries ComfyUI quantization metadata - and refuses any layout it can't reverse
# @markdown exactly. `as-is` writes the packed integers as weights: noise, inspection only.
IF_ALREADY_QUANTIZED = "unpack and convert"  # @param ["unpack and convert", "stop", "convert as-is (unsafe)"]
IF_TARGET_TOO_HIGH = "drop it"  # @param ["drop it", "build it anyway", "stop"]

# @markdown ---
# @markdown `llama-quantize` only knows the architectures city96's patch adds.
# @markdown
# @markdown `stand-in` relabels the file as a known family just long enough to quantize, which
# @markdown gets K-quants (Q4_K_M …). Every tensor the base file holds in F32 - everything this
# @markdown notebook protects - is copied back afterwards, so the borrowed family's rules can't
# @markdown quantize them. `python` quantizes here instead: Q8_0, Q5_1, Q5_0, Q4_1, Q4_0 only.
IF_ARCH_UNKNOWN = "stand-in arch"  # @param ["stand-in arch", "quantize in python", "drop the quants"]
STAND_IN_ARCH = "auto"  # @param ["auto", "flux", "sd3", "aura", "ltxv", "hyvid", "wan", "hidream", "cosmos", "lumina2"]

PROGRESS = "summary"  # @param ["quiet", "summary", "every tensor"]

# @markdown ---
# @markdown ComfyUI-GGUF repos to read architectures from, comma separated. The first one's
# @markdown `lcpp.patch` builds llama-quantize; the others only add families.
# @markdown `molbal/ComfyUI-GGUF` adds Krea 2 (`krea2`), Qwen-Image 2.1, Ideogram and others.
GGUF_NODE_REPOS = "city96/ComfyUI-GGUF, molbal/ComfyUI-GGUF, RealRebelAI/ComfyUI-GGUF_KREA-2"  # @param ["city96/ComfyUI-GGUF, molbal/ComfyUI-GGUF, RealRebelAI/ComfyUI-GGUF_KREA-2", "city96/ComfyUI-GGUF"] {allow-input: true}
GGUF_NODE_REF = "main"  # @param {type:"string"}
GGUF_NODE_REPO = [r.strip() for r in GGUF_NODE_REPOS.split(",") if r.strip()][0]
# @markdown The GGUF node you will load the files with. The architecture name written into
# @markdown each file is one this loader accepts, and the final check loads with it.
# @markdown `auto` picks the first repo above whose loader accepts the detected family.
LOAD_WITH = "auto"  # @param ["auto", "city96/ComfyUI-GGUF", "molbal/ComfyUI-GGUF", "RealRebelAI/ComfyUI-GGUF_KREA-2"] {allow-input: true}

CONFIDENT_AT = 1.0  # @param {type:"slider", min:0.5, max:1.0, step:0.05}
# @markdown `pig` makes ComfyUI-GGUF detect the family from tensor names at load time. It's
# @markdown only used when the chosen loader's own detection really recognises the weights.
DEFER_WHEN_UNSURE = True  # @param {type:"boolean"}

# @markdown ---
# @markdown Keep tensors the checkpoint itself stores in F32 (inside an otherwise bf16/fp16
# @markdown file) as F32 in every output. Model authors do this on purpose for the layers that
# @markdown can't take the rounding - Krea 2's input, output, timestep and text projections.
KEEP_SOURCE_F32 = True  # @param {type:"boolean"}

# @markdown llama.cpp for image-model quants (city96's patch targets this tag), and for LLMs.
LLAMA_TAG = "b3962"  # @param {type:"string"}
LLM_LLAMA_REF = "master"  # @param {type:"string"}

In [ ]:
# @title 1e · Checks { display-mode: "form" }

# @markdown Compare tensor names and shapes against the original checkpoint. This is what
# @markdown finds dropped and mis-shaped weights, so leave it on.
COMPARE_TO_SOURCE = True  # @param {type:"boolean"}
# @markdown Check each quant kept the same tensors and shapes as the file it came from.
COMPARE_QUANTS_TO_BASE = True  # @param {type:"boolean"}
# @markdown Look for NaN and Inf. More tensors is slower, and these files are big.
NAN_CHECK = "normal"  # @param ["off", "quick", "normal", "thorough", "every tensor"]

# @markdown ---
# @markdown Compare the *values* in each gguf against the checkpoint they came from,
# @markdown tensor by tensor, grouped by the part of the network each belongs to. Names
# @markdown and shapes matching proves nothing about whether the numbers survived - a
# @markdown rescaled, transposed or half-converted weight passes every structural check
# @markdown there is. This is how you find which part of the model broke rather than
# @markdown just that it broke. Tensors compared per role; 0 turns it off.
DIFF_SAMPLE = 4  # @param {type:"slider", min:0, max:16, step:1}

# @markdown ---
# @markdown Tie every file produced to the exact checkpoint it came from. The source is
# @markdown fingerprinted, the fingerprint is written into each gguf, and every later pass
# @markdown checks it - so a quant can't be cut from someone else's base file, a stale gguf
# @markdown in the output folder can't be mistaken for this one, and a file that doesn't
# @markdown match what's in 1b fails the checks instead of being shipped.
# @markdown
# @markdown `sampled` reads the header plus evenly spaced chunks, a few seconds even on a
# @markdown large checkpoint. `full` reads every byte, which is exact but slower. A finetune
# @markdown differs in every block, so `sampled` still tells two checkpoints apart.
SOURCE_STAMP = "sampled"  # @param ["sampled", "full", "off"]

# @markdown ---
# @markdown Weights that must stay F32. `auto` takes whatever the detected family
# @markdown protects and adds anything this checkpoint's own tensor names imply on top -
# @markdown a family template can be older than the model in front of it. Type comma
# @markdown separated patterns to override it outright.
KEEP_IN_F32 = "auto"  # @param {type:"string"}

# @markdown ---
# @markdown Files that have to ship next to the gguf. A diffusion gguf is only the
# @markdown transformer, so on its own it won't run.
NEED_VAE = False  # @param {type:"boolean"}
NEED_TEXT_ENCODER = False  # @param {type:"boolean"}
NEED_CONNECTORS = False  # @param {type:"boolean"}
NEED_OTHER_FILES = "none"  # @param {type:"string"}

# @markdown ---
# @markdown ---
# @markdown Load every diffusion gguf in ComfyUI with the GGUF node from 1d, the way
# @markdown `Unet Loader (GGUF)` does. Fails on missing keys (those weights stay random, which
# @markdown is noise), compares every loaded weight against the checkpoint through the node's
# @markdown own dequantizer, and runs one forward pass of each quant against the base file.
# @markdown Runs on CPU, so no GPU is needed.
COMFYUI_CHECK = "weights + forward pass"  # @param ["weights + forward pass", "weights only", "off"]
COMFYUI_REF = "master"  # @param {type:"string"}

# @markdown `strict` turns warnings into failures.
STRICTNESS = "normal"  # @param ["normal", "strict"]
# @markdown Save the results to `OUT_DIR/verification_report.md`.
SAVE_REPORT = True  # @param {type:"boolean"}
# @markdown Push the files and the report to a Hub repo. Empty keeps everything local.
UPLOAD_REPO = ""  # @param {type:"string"}

### 1f. Apply & check

In [ ]:
import os, re, json, struct, shutil, subprocess, sys, time
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode, unquote

BROWSER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

TASKS = {
    "auto-detect":                   ("auto", "auto"),
    "text / chat LLM":               ("llm", "text"),
    "vision (VQA, captioning, OCR)": ("llm", "vision"),
    "audio-in LLM":                  ("llm", "audio"),
    "embedding / reranker":          ("llm", "embedding"),
    "image generation":              ("diffusion", "image"),
    "video generation":              ("diffusion", "video"),
}

DTYPE_BITS = {"F64": 64, "I64": 64, "F32": 32, "I32": 32, "BF16": 16, "F16": 16,
              "I16": 16, "F8_E4M3": 8, "F8_E5M2": 8, "I8": 8, "U8": 8,
              "I4": 4, "U4": 4, "BOOL": 1}

# Which tensors are quantization bookkeeping rather than weights.
#
# A suffix alone is not evidence. `.scale` is also what RMSNorm calls its weight (Krea 2,
# Gemma, Chroma, ...), and treating those as quantization scales made a clean bf16 checkpoint
# look "already quantized" and get refused. A tensor is a scale when it pairs with a weight
# stored in 8 bits or fewer, or when its name can only mean quantization.
LOWBIT_DTYPES = {"I8", "U8", "I4", "U4", "F8_E4M3", "F8_E5M2", "F8_E8M0", "F4"}
SCALE_SUFFIXES = ("_scale_inv", ".scale_inv", ".weight_scale_2", ".weight_scale",
                  "_weight_scale", ".scale_weight", ".weight_s_rel", ".weight_s_channel",
                  ".weight_codebook", "_scale", ".scales", ".scale", ".absmax", ".wscale",
                  ".qscale")
# only ever quantization metadata - dropped from the output whatever they pair with
QUANT_AUX_SUFFIXES = (".comfy_quant", ".input_scale", ".scale_input", ".pre_quant_scale",
                      ".weight_scale_2", "scaled_fp8")
PACKED_MARKERS = ("qweight", "qzeros", "g_idx", "quant_map")

# a real diffusion weight sits in here. Raw int8 codes land around 10-80, which is how
# a missing scale is caught without knowing anything about the architecture.
SANE_STD = (5e-4, 1.0)

# which part of the network a tensor belongs to, first match wins. Used to summarise a
# value comparison by role - one role sitting far worse than the rest says which stage
# of the forward pass is broken, which is more useful than any single tensor name.
ROLES = [
    ("q/k norm",    r"(q_norm|k_norm|norm_q|norm_k|query_norm|key_norm)"),
    ("attention",   r"(attn|attention|qkv|to_q|to_k|to_v|to_out)"),
    ("modulation",  r"(modulation|adaln|ada_ln|scale_shift|\bmod\b|gate)"),
    ("positional",  r"(rope|freqs|pos_embed|position)"),
    ("embedding",   r"(embed|embedder|patch|x_in|img_in|txt_in|cap_|pad_token)"),
    ("output",      r"(final|proj_out|\bhead\b|out_layer|unpatch)"),
    ("feedforward", r"(mlp|ffn|feed_forward|\bw[123]\b|linear[12])"),
    ("norm",        r"(norm|layernorm|rmsnorm|ln_)"),
]

# structural read of a checkpoint, used when no family name fits
BLOCK_HINTS = {
    "double_blocks": "flux-style dual stream",
    "single_blocks": "flux-style single stream",
    "joint_blocks": "sd3-style joint",
    "transformer_blocks": "diffusers-style DiT",
    "input_blocks": "unet (sd1/sdxl era)",
    "output_blocks": "unet (sd1/sdxl era)",
    "middle_block": "unet (sd1/sdxl era)",
    "noise_refiner": "lumina2 / z-image family",
    "context_refiner": "lumina2 / z-image family",
}

PREQUANT_HINTS = ["int8", "int4", "fp8", "f8_", "convrot", "quanto",
                  "nf4", "awq", "gptq", "svdq", "scaled", "nvfp4", "mxfp4"]

# precision-sensitive in any architecture that has them, used when deriving a template
GENERIC_HIPREC = ["modulation", "scale_shift", "pos_embed", "emb_pos", "pos_embedder",
                  "position_embedding", "register", "logit_scale", "freqs"]

FALLBACK_IMG = {"flux", "sd1", "sdxl", "sd3", "aura", "hidream", "cosmos",
                "ltxv", "hyvid", "wan", "lumina2", "qwen_image"}
FALLBACK_TXT = {"t5", "t5encoder", "llama", "qwen2vl", "qwen3", "qwen3vl", "gemma3"}

# Krea 2, from the published architecture (ComfyUI comfy/ldm/krea2, and molbal's template).
# Only used when none of the repos in 1d defines the family. `^` anchors at the start.
LOCAL_TEMPLATES = {
    "krea2": {
        "class": "builtin", "repo": "built-in",
        "keys_detect": [("blocks.0.attn.wq.weight", "txtfusion.projector.weight", "first.weight")],
        "keys_hiprec": ["^first.", "^last.", "^tproj.", "^tmlp.", "^txtmlp.",
                        "^txtfusion.projector."],
        "keys_banned": [], "keys_noquant": [], "shape_fix": False,
    },
}

# quant types the patched llama-quantize refuses for image models (city96 lcpp.patch)
LLAMA_IMG_REJECTS = {"IQ1_S", "IQ1_M", "IQ2_XXS", "IQ2_XS", "IQ2_S", "IQ2_M", "IQ3_XXS",
                     "IQ3_S", "IQ3_M", "IQ4_NL", "IQ4_XS"}
NEAREST_QUANT = {"Q4_K_M": "Q4_1", "Q4_K_S": "Q4_0", "Q5_K_M": "Q5_1", "Q5_K_S": "Q5_0",
                 "Q6_K": "Q8_0", "Q3_K_M": "Q4_0", "Q3_K_S": "Q4_0", "Q2_K": "Q4_0",
                 "IQ4_NL": "Q4_1", "IQ4_XS": "Q4_0"}


def matches_any(name, patterns):
    """Pattern rules the ComfyUI-GGUF forks use: a plain pattern matches anywhere, `^p`
    only at the start, `*`/`?` are shell wildcards. Plain substring matching turns `^first.`
    into a pattern nothing contains, which silently protects nothing."""
    from fnmatch import fnmatchcase
    for p in patterns:
        if not p:
            continue
        anchored = p.startswith("^")
        body = p[1:] if anchored else p
        if "*" in body or "?" in body:
            if fnmatchcase(name, body if anchored else f"*{body}*"):
                return True
        elif anchored:
            if name.startswith(body):
                return True
        elif body in name:
            return True
    return False

In [ ]:
# @title listify · turn a comma separated box into a list

def listify(value):
    if not value or value.strip().lower() in ("none", "no", "all", ""):
        return []
    return [v.strip() for v in value.replace(",", " ").split() if v.strip()]

In [ ]:
# @title get_token · form field, then env var, then Colab secret

def get_token(name, given):
    tok = given.strip()
    if tok:
        return tok
    tok = os.environ.get(name, "")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

In [ ]:
MODE, SUBTASK = TASKS[TASK]

DO_CONVERT = JOB in ("convert and check", "convert only")
DO_CHECK = JOB in ("convert and check", "check only")

WORK = Path(WORK_DIR).resolve(); WORK.mkdir(parents=True, exist_ok=True)
OUT = Path(OUT_DIR).resolve(); OUT.mkdir(parents=True, exist_ok=True)
GG = WORK / "ComfyUI-GGUF"
LCPP = WORK / "llama.cpp"            # patched, for image-model quants
LCPP_LLM = WORK / "llama.cpp-llm"    # current, for LLM conversion and quants
COMFY = WORK / "ComfyUI"
STASH = WORK / "stash"; STASH.mkdir(exist_ok=True)
MODEL_DIR = WORK / "model"; MODEL_DIR.mkdir(exist_ok=True)

HF_TOK = get_token("HF_TOKEN", HF_TOKEN)
CIV_TOK = get_token("CIVITAI_TOKEN", CIVITAI_TOKEN)

URLS = [u for u in re.split(r"[,\s]+", SRC_URL.strip()) if u]
if DO_CONVERT and SRC_FROM == "download link" and not URLS:
    raise ValueError("SRC_FROM is 'download link' but SRC_URL is empty - fill it in in 1b")
if DO_CONVERT and SRC_FROM == "local path" and not SRC_PATH.strip():
    raise ValueError("SRC_FROM is 'local path' but SRC_PATH is empty - fill it in in 1b")
if DO_CONVERT and SRC_FROM == "Hugging Face repo" and not SRC_REPO.strip():
    raise ValueError("SRC_FROM is 'Hugging Face repo' but SRC_REPO is empty - fill it in in 1b")

if MODE == "auto":
    single = (SRC_FILE.strip().endswith(".safetensors")
              or SRC_PATH.strip().endswith(".safetensors")
              or any(u.endswith(".safetensors") for u in URLS))
    MODE = "diffusion" if single else "llm"
    print("auto-detect -> pipeline:", MODE)

TARGET = RUNTIME
if TARGET == "auto":
    TARGET = "ComfyUI (GGUF nodes)" if MODE == "diffusion" else "llama.cpp / Ollama"

QUANTS = [q for on, q in [(WANT_8BIT, RECIPE_8BIT), (WANT_4BIT, RECIPE_4BIT)] if on]
QUANTS += [q.upper() for q in listify(EXTRA_QUANTS)]
QUANTS = list(dict.fromkeys(QUANTS))
EXPECT = (["mix"] if WANT_MIX else []) + QUANTS
if DO_CONVERT and not EXPECT:
    raise ValueError("nothing ticked in 1c - there'd be nothing to build")

WANT_MMPROJ = {"auto": SUBTASK in ("vision", "audio"), "yes": True, "no": False}[EXPECT_MMPROJ]
NEED_LLAMA = bool(QUANTS) or MODE == "llm"

COMPANIONS = []
if NEED_VAE:
    COMPANIONS.append(("VAE", ["*vae*"]))
if NEED_TEXT_ENCODER:
    COMPANIONS.append(("text encoder", ["*text_encoder*", "*t5*", "*gemma*", "*clip*", "*umt5*"]))
if NEED_CONNECTORS:
    COMPANIONS.append(("connectors", ["*connector*"]))
for pat in listify(NEED_OTHER_FILES):
    COMPANIONS.append((pat, [pat]))

F32_OVERRIDE = None if KEEP_IN_F32.strip().lower() == "auto" else \
    [p.strip() for p in KEEP_IN_F32.split(",") if p.strip()]
F32_PATTERNS = F32_OVERRIDE or []

NAN_SAMPLE = {"off": None, "quick": 20, "normal": 40, "thorough": 200, "every tensor": 0}[NAN_CHECK]
WANT_5D = SUBTASK == "video"
STRICT = STRICTNESS == "strict"

print("job      :", JOB)
print("pipeline :", MODE, "/", SUBTASK)
print("runtime  :", TARGET)
print("building :", ", ".join(EXPECT) + (" + mmproj" if WANT_MMPROJ else ""))
print("arch     :", ARCH_MODE + (f" -> {ARCH_NAME}" if ARCH_NAME.strip() else ""))
print("out      :", OUT)
print("tokens   : hf=" + ("set" if HF_TOK else "none") + "  civitai=" + ("set" if CIV_TOK else "none"))

## Setup

In [ ]:
!apt-get -qq install -y aria2 > /dev/null 2>&1
!pip install -q "gguf>=0.17" safetensors requests "huggingface_hub>=0.26" tqdm

import ast, requests
import numpy as np
import gguf
import torch

HAVE_ARIA2 = shutil.which("aria2c") is not None
RAW = (f"https://raw.githubusercontent.com/{GGUF_NODE_REPO.strip()}"
       f"/{GGUF_NODE_REF.strip() or 'main'}")
print("aria2c:", "ready" if HAVE_ARIA2 else "missing, will fall back to single stream")
print("gguf-py:", getattr(gguf, "__version__", "?"), " torch:", torch.__version__)

## Download

In [ ]:
# @title add_query_param · add or replace one parameter on a URL

def add_query_param(url, key, value):
    parts = urlsplit(url)
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    query[key] = value
    return urlunsplit(parts._replace(query=urlencode(query)))

In [ ]:
# @title auth_for · attach the right credential for the host

def auth_for(url):
    """Hugging Face wants an Authorization header. Civitai wants a token query parameter,
    because its download URL redirects to a pre-signed CDN link and a second credential on
    that request gets rejected."""
    headers = {"User-Agent": BROWSER_UA}
    host = urlsplit(url).netloc.lower()

    if "huggingface.co" in host or "hf.co" in host:
        if HF_TOK:
            headers["Authorization"] = f"Bearer {HF_TOK}"
    elif "civitai" in host:
        if CIV_TOK:
            url = add_query_param(url, "token", CIV_TOK)

    return url, headers

In [ ]:
# @title filename_for · ask the server what the file is actually called

def filename_for(url, headers, override=""):
    if override.strip():
        return override.strip()

    try:
        r = requests.get(url, headers={**headers, "Range": "bytes=0-0"},
                         stream=True, timeout=60, allow_redirects=True)
        cd = r.headers.get("content-disposition", "")
        r.close()
        m = re.search(r"filename\*=UTF-8''([^;]+)", cd) or re.search(r'filename="?([^";]+)', cd)
        if m:
            return unquote(m.group(1)).strip()
    except Exception as e:
        print("  couldn't ask the server for a name:", e)

    return unquote(Path(urlsplit(url).path).name) or "download.bin"

In [ ]:
# @title download · one URL to disk, skipping anything already there

def download(url, dest_dir, override=""):
    dest_dir = Path(dest_dir); dest_dir.mkdir(parents=True, exist_ok=True)
    url, headers = auth_for(url)
    name = filename_for(url, headers, override)
    dest = dest_dir / name

    # aria2 leaves a .aria2 control file next to a transfer it hasn't finished. Without
    # this the next run sees a big file, calls it done, and converts half a checkpoint.
    control = dest.with_name(dest.name + ".aria2")
    if dest.exists() and dest.stat().st_size > 0 and not control.exists():
        print(f"  have {name} ({dest.stat().st_size / 2**30:.2f} GB)")
        return dest
    if control.exists():
        print(f"  {name} was left part-finished, resuming")

    print(f"  {name}")
    if HAVE_ARIA2:
        command = [
            "aria2c", "--console-log-level=warn",
            "-c", "-x", str(CONNECTIONS), "-s", str(CONNECTIONS), "-k", "1M",
            "--max-tries=5", "--retry-wait=5",
            "--allow-overwrite=true", "--auto-file-renaming=false",
            "--summary-interval=5", f"--user-agent={BROWSER_UA}",
            *[f"--header={k}: {v}" for k, v in headers.items() if k != "User-Agent"],
            url, "-d", str(dest_dir), "-o", name,
        ]
        proc = subprocess.Popen(command, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            if line.strip():
                print("   ", line.rstrip())
        if proc.wait() != 0:
            raise RuntimeError(f"aria2c exited {proc.returncode} on {name}")
    else:
        with requests.get(url, headers=headers, stream=True, timeout=120,
                          allow_redirects=True) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(1 << 22):
                    fh.write(chunk)

    print(f"  done, {dest.stat().st_size / 2**30:.2f} GB")
    return dest

In [ ]:
SRC = None
COMPANION_FILES = []


def to_safetensors(path):
    """.ckpt/.pt/.pth/.bin -> .safetensors next to it, so every later step reads one format."""
    path = Path(path)
    if path.suffix.lower() not in (".ckpt", ".pt", ".pth", ".bin"):
        return path
    out = path.with_suffix(".safetensors")
    if out.exists():
        return out
    from safetensors.torch import save_file
    print(f"  {path.name} is a pickle checkpoint, rewriting as safetensors")
    sd = torch.load(str(path), map_location="cpu", weights_only=True)
    for sub in ("state_dict", "model", "module"):
        if isinstance(sd, dict) and sub in sd and isinstance(sd[sub], dict):
            sd = sd[sub]
    sd = {k: v.contiguous() for k, v in sd.items() if isinstance(v, torch.Tensor)}
    save_file(sd, str(out))
    return out


def pick_diffusers_part(folder):
    """A diffusers repo holds the transformer, the VAE and the text encoders side by side.
    Converting the folder as one checkpoint mixes all of them into one gguf."""
    folder = Path(folder)
    if not (folder / "model_index.json").exists():
        return folder
    for part in ("transformer", "unet", "prior", "decoder"):
        if (folder / part).is_dir() and any((folder / part).glob("*.safetensors")):
            print(f"  diffusers layout - converting only {part}/")
            return folder / part
    return folder


if SRC_FROM == "download link" and URLS:
    got = [download(u, MODEL_DIR) for u in URLS]
    SRC = to_safetensors(got[0])
    COMPANION_FILES = got[1:]
    for extra in COMPANION_FILES:
        shutil.copy2(extra, OUT / extra.name)

elif SRC_FROM == "local path" and SRC_PATH.strip():
    SRC = Path(SRC_PATH).expanduser().resolve()
    if not SRC.exists():
        raise FileNotFoundError(SRC)
    SRC = to_safetensors(SRC) if SRC.is_file() else SRC

elif SRC_FROM == "Hugging Face repo" and SRC_REPO.strip():
    from huggingface_hub import snapshot_download, hf_hub_download
    if SRC_FILE.strip():
        SRC = to_safetensors(hf_hub_download(SRC_REPO.strip(), SRC_FILE.strip(),
                                             revision=SRC_REVISION.strip() or "main",
                                             local_dir=str(MODEL_DIR), token=HF_TOK or None))
    else:
        allow = None
        if MODE == "diffusion":
            # a diffusers repo: the transformer and the configs, not the 10+ GB of encoders
            allow = ["*.json", "transformer/*", "unet/*", "*.safetensors"]
            from huggingface_hub import list_repo_files
            names = list_repo_files(SRC_REPO.strip(), revision=SRC_REVISION.strip() or "main",
                                    token=HF_TOK or None)
            if "model_index.json" in names:
                allow = ["*.json", "transformer/*", "unet/*"]
        SRC = Path(snapshot_download(SRC_REPO.strip(),
                                     revision=SRC_REVISION.strip() or "main",
                                     allow_patterns=allow,
                                     local_dir=str(MODEL_DIR), token=HF_TOK or None))

if SRC is not None and SRC.is_dir() and MODE == "diffusion":
    SRC = pick_diffusers_part(SRC)

print("\nsource:", SRC or "none (check only)")
if COMPANION_FILES:
    print("companions:", ", ".join(p.name for p in COMPANION_FILES))

## The checkpoint

In [ ]:
# @title read_header · tensor list from a safetensors file, no weights loaded

def read_header(path):
    with open(path, "rb") as fh:
        (n,) = struct.unpack("<Q", fh.read(8))
        return {k: v for k, v in json.loads(fh.read(n)).items() if k != "__metadata__"}

In [ ]:
# @title read_field · one metadata value out of a gguf, safe on numpy 2

def read_field(reader, key):
    """gguf-py hands back size-1 arrays and numpy 2 refuses to int() those, which is why
    upstream's fix_5d_tensors.py dies with 'only 0-dimensional arrays can be converted
    to Python scalars'."""
    field = reader.get_field(key) if hasattr(reader, "get_field") else reader.fields.get(key)
    if field is None:
        return None

    part = field.parts[field.data[-1]]
    if field.types and field.types[0] == gguf.GGUFValueType.STRING:
        return bytes(part).decode("utf-8", errors="replace")
    return part.tolist()[0] if hasattr(part, "tolist") else part

In [ ]:
# @title is_complete · did the file finish being written

def safetensors_is_complete(path):
    """The header says where every tensor ends, so the file has a size it must be. A
    download or copy that was cut short is shorter than that, and read_header can't see it
    because it only reads the front."""
    path = Path(path)
    with open(path, "rb") as fh:
        (n,) = struct.unpack("<Q", fh.read(8))
        header = json.loads(fh.read(n))

    end = 0
    for key, meta in header.items():
        if key != "__metadata__":
            end = max(end, int(meta["data_offsets"][1]))

    need = 8 + n + end
    have = path.stat().st_size
    return have >= need, need, have


def gguf_is_complete(path):
    """A run killed part way through write_tensors_to_file leaves a gguf whose header
    promises more data than the file holds. gguf-py mmaps it and hands back short reads
    rather than complaining, so the size is checked directly.

    data_offset is absolute in current gguf-py and was relative in older ones, so both
    readings are taken and the larger requirement wins."""
    path = Path(path)
    have = path.stat().st_size
    try:
        reader = gguf.GGUFReader(str(path))
        need = max([int(t.data_offset) + int(t.n_bytes) for t in reader.tensors] or [0])
        need = max(need, sum(int(t.n_bytes) for t in reader.tensors))
        del reader
    except Exception as e:
        return False, f"it can't be opened as a gguf at all ({type(e).__name__})"

    if have >= need:
        return True, f"{have / 2**30:.2f} GB, all its header describes"
    return False, (f"{have / 2**30:.2f} GB of the {need / 2**30:.2f} GB its header "
                   f"describes")

In [ ]:
# @title source_fingerprint · a short id for the checkpoint in 1b

def file_digest(path, method):
    """`full` reads every byte. `sampled` reads the front, the back and evenly spaced
    chunks in between, which is seconds rather than minutes on a large checkpoint and still
    separates two models - a finetune differs in every block, not in one place."""
    import hashlib

    path = Path(path)
    size = path.stat().st_size
    h = hashlib.sha256()
    # deliberately not the filename: the same checkpoint arrives under different names
    # from Civitai, HF and a local copy, and all three should fingerprint the same
    h.update(f"{size}:".encode())

    with open(path, "rb") as fh:
        if method == "full":
            for chunk in iter(lambda: fh.read(1 << 24), b""):
                h.update(chunk)
        else:
            span = 1 << 22                       # 4 MB per probe
            step = max(span, size // 48)
            for pos in range(0, size, step):
                fh.seek(pos)
                h.update(fh.read(span))
            fh.seek(max(0, size - span))         # always include the tail
            h.update(fh.read(span))

    return h.hexdigest()


def source_fingerprint(source, method):
    """One id over every safetensors file in the source, so a multi-file model can't be
    confused with a subset of itself."""
    if method == "off" or source is None:
        return {}

    import hashlib

    source = Path(source)
    files = sorted(source.rglob("*.safetensors")) if source.is_dir() else [source]
    if not files:
        return {}

    size = sum(f.stat().st_size for f in files)
    if len(files) == 1:
        digest = file_digest(files[0], method)
    else:
        roll = hashlib.sha256()
        for f in files:
            roll.update(file_digest(f, method).encode())
        digest = roll.hexdigest()

    return {"file": source.name, "size": size, "hash": digest, "method": method}


def write_stamp(writer):
    """Written into every file produced. This is what later passes compare against."""
    if not SRC_FP:
        return
    writer.add_string("general.source.file", str(SRC_FP["file"]))
    writer.add_string("general.source.hash", str(SRC_FP["hash"]))
    writer.add_string("general.source.hash_method", str(SRC_FP["method"]))
    writer.add_uint64("general.source.size", int(SRC_FP["size"]))


def read_stamp(path):
    reader = gguf.GGUFReader(str(path))
    stamp = {name: read_field(reader, f"general.source.{name}")
             for name in ("file", "size", "hash", "hash_method")}
    del reader
    return {k: v for k, v in stamp.items() if v is not None}


def stamp_matches(path):
    """`match`, `different`, `unstamped` or `unchecked`, with something to print."""
    if not SRC_FP:
        return "unchecked", "SOURCE_STAMP is off, nothing ties this file to a checkpoint"

    stamp = read_stamp(path)
    if not stamp.get("hash"):
        return "unstamped", ("no source stamp - made before this check existed, or by "
                             "another tool")
    if stamp.get("hash_method") != SRC_FP["method"]:
        return "unchecked", (f"stamped with the {stamp.get('hash_method')!r} method, this "
                             f"run is using {SRC_FP['method']!r} - not comparable")
    if stamp["hash"] == SRC_FP["hash"]:
        return "match", f"{stamp.get('file')} {stamp['hash'][:12]}"

    return "different", (f"stamped {stamp.get('file')} {stamp['hash'][:12]}, the checkpoint "
                         f"in 1b is {SRC_FP['file']} {SRC_FP['hash'][:12]}")

In [ ]:
# @title strip_prefix · what the converter keeps, and what it quietly throws away

def strip_prefix(keys):
    """Same choice of prefix as city96's convert.py, but only removed from the front of a
    key - str.replace() also cut it out of the middle of names that happen to contain it."""
    prefix = None
    for pfx in ("model.diffusion_model.", "model."):
        if any(k.startswith(pfx) for k in keys):
            prefix = pfx
            break
    if prefix is None and keys and all(k.startswith("net.") for k in keys):
        prefix = "net."
    if prefix is None:
        return {k: k for k in keys}, []

    kept, dropped = {}, []
    for k in keys:
        if k.startswith(prefix):
            kept[k] = k[len(prefix):]
        else:
            dropped.append(k)
    return kept, dropped

In [ ]:
# @title find_scale_pairs · match each low-bit weight to its scale tensor

def find_scale_pairs(shapes):
    """{weight: scale} for weights stored in 8 bits or fewer, plus anything bit-packed in a
    way only its original library can undo (GPTQ/AWQ qweight/qzeros/g_idx).

    `shapes` is a safetensors header {name: {"dtype", "shape"}}. The weight's own dtype is
    what makes a `.scale` a quantization scale - an RMSNorm `.scale` next to a bf16 weight
    is a weight, and treating it as a scale is how a clean checkpoint got refused."""
    names = set(shapes)
    pairs = {}
    packed = sorted(n for n in names if any(t in n.lower() for t in PACKED_MARKERS))

    for key in names:
        if key.endswith(".scale_weight"):                       # legacy ComfyUI scaled_fp8
            cands = [key[: -len(".scale_weight")] + ".weight"]
        else:
            cands = []
            for suffix in SCALE_SUFFIXES:
                if key.endswith(suffix) and len(key) > len(suffix):
                    stem = key[: -len(suffix)]
                    cands = [stem, stem + ".weight"]
                    break
        for cand in cands:
            if (cand in names and cand != key
                    and shapes[cand]["dtype"] in LOWBIT_DTYPES
                    and shapes[key]["dtype"] not in LOWBIT_DTYPES - {"F8_E4M3", "F8_E8M0"}):
                # nvfp4 keeps a fp8 block scale and a second fp32 one - the first wins here,
                # and that layout is only ever decoded through ComfyUI anyway
                if cand not in pairs or key.endswith(".weight_scale"):
                    pairs[cand] = key
                break

    return pairs, packed


def is_quant_aux(key):
    return key.endswith(QUANT_AUX_SUFFIXES)

In [ ]:
# @title read_comfy_quant · ComfyUI's per-layer quantization metadata

def read_comfy_quant(files):
    """ComfyUI's mixed-precision format stores, next to each quantized layer, a uint8 tensor
    `<layer>.comfy_quant` holding JSON: {"format": "float8_e4m3fn" | "int8_tensorwise" |
    "nvfp4" | "mxfp8" | ..., "convrot": true, ...}. It is the only reliable statement of how
    the layer was encoded - int8 with convrot stores Hadamard-rotated weights, nvfp4 packs two
    4-bit floats per byte with swizzled block scales, fp8 is sometimes stored as raw uint8.
    Returns {weight key: config}."""
    from safetensors import safe_open
    layers = {}
    for path in files:
        with safe_open(str(path), framework="pt", device="cpu") as f:
            for key in f.keys():
                if key.endswith(".comfy_quant"):
                    try:
                        conf = json.loads(bytes(f.get_tensor(key).to(torch.uint8).tolist()))
                    except Exception as e:
                        conf = {"format": f"unreadable ({e})"}
                    layers[key[: -len("comfy_quant")] + "weight"] = conf
    return layers

In [ ]:
SRC_SHAPES = None
SRC_FP = {}
SRC_LABEL = str(SRC) if SRC else "not given"
SRC_FILES, SRC_INDEX = [], {}
COMFY_LAYERS, SRC_AUX = {}, set()
SOURCE_HAS_F32_INTENT = False

if SRC is not None:
    SRC_FILES = sorted(SRC.rglob("*.safetensors")) if SRC.is_dir() else [SRC]

    for f in SRC_FILES:
        ok, need, have = safetensors_is_complete(f)
        if not ok:
            raise RuntimeError(
                f"{f.name} is {have / 2**30:.2f} GB but its header describes "
                f"{need / 2**30:.2f} GB - the download or copy didn't finish. Delete it "
                f"and run 1b again rather than converting a truncated checkpoint.")
    if SRC_FILES:
        print(f"{len(SRC_FILES)} file(s), all the length their headers say they should be")
        SRC_SHAPES = {}
        for f in SRC_FILES:
            head = read_header(f)
            SRC_SHAPES.update(head)
            SRC_INDEX.update({k: f for k in head})
        print(f"{len(SRC_SHAPES)} tensors across {len(SRC_FILES)} file(s)")

    SRC_FP = source_fingerprint(SRC, SOURCE_STAMP)
    if SRC_FP:
        print(f"source id: {SRC_FP['hash'][:16]} ({SRC_FP['method']}, "
              f"{SRC_FP['size'] / 2**30:.2f} GB) - written into every file produced")
    else:
        print("[!] SOURCE_STAMP is off, so nothing produced here will record which "
              "checkpoint it came from")

if SRC_SHAPES:
    if SRC.is_dir() and (SRC / "config.json").exists():
        try:
            cfg = json.loads((SRC / "config.json").read_text())
            if "_diffusers_version" in cfg:
                print(f"[!] diffusers-format transformer ({cfg.get('_class_name', '?')}). "
                      f"ComfyUI renames diffusers keys for some families only; for the rest the "
                      f"weights load under the wrong names and stay random. The ComfyUI check "
                      f"at the end settles it - a repack with ComfyUI key names is safer.")
        except Exception:
            pass

    dtypes = {}
    for v in SRC_SHAPES.values():
        dtypes[v["dtype"]] = dtypes.get(v["dtype"], 0) + 1
    print("dtypes:", ", ".join(f"{k}={v}" for k, v in sorted(dtypes.items(), key=lambda x: -x[1])))

    over_4d = [k for k, v in SRC_SHAPES.items() if len(v["shape"]) > 4]
    if over_4d:
        print(f"over 4 dimensions: {len(over_4d)} -> stashed and restored after quantizing")

    COMFY_LAYERS = read_comfy_quant(SRC_FILES)
    SRC_PAIRS, SRC_PACKED = find_scale_pairs(SRC_SHAPES)
    for w in COMFY_LAYERS:                  # ComfyUI layers are decoded as a whole, not paired
        SRC_PAIRS.pop(w, None)

    # everything that is bookkeeping, not a weight: never written to the gguf
    for w in COMFY_LAYERS:
        stem = w[: -len("weight")]
        SRC_AUX |= {k for k in SRC_SHAPES
                    if k.startswith(stem) and k not in (w, stem + "bias")
                    and "." not in k[len(stem):]}
    SRC_AUX |= set(SRC_PAIRS.values())
    SRC_AUX |= {k for k in SRC_SHAPES if is_quant_aux(k)}
    SCALES = sorted(SRC_AUX)

    wd = {}
    for k, v in SRC_SHAPES.items():
        if k in SRC_AUX or len(v["shape"]) < 2:
            continue
        wd[v["dtype"]] = wd.get(v["dtype"], 0) + 1
    main_dtype = max(wd, key=wd.get) if wd else None
    SRC_BITS = DTYPE_BITS.get(main_dtype)
    if COMFY_LAYERS:
        fmts = {}
        for conf in COMFY_LAYERS.values():
            tag = conf.get("format", "?") + ("+convrot" if conf.get("convrot") else "")
            fmts[tag] = fmts.get(tag, 0) + 1
        bits = [{"nvfp4": 4, "convrot_w4a4": 4, "asym_w4a8_int8": 4, "w6a8_int8": 6}.get(
                c.get("format"), 8) for c in COMFY_LAYERS.values()]
        share = len(COMFY_LAYERS) / max(1, sum(wd.values()))
        print(f"\n[!] ComfyUI-quantized checkpoint: {len(COMFY_LAYERS)} layer(s) "
              f"({', '.join(f'{k}={v}' for k, v in fmts.items())}), "
              f"{share:.0%} of the weights")
        print("    They will be decoded with ComfyUI's own dequantizer, never guessed.")
        if share >= 0.5:
            SRC_BITS = max(bits)
            main_dtype = f"comfy {max(fmts, key=fmts.get)}"
    print("weight dtypes:", ", ".join(f"{k}={v}" for k, v in
                                      sorted(wd.items(), key=lambda x: -x[1])) or "none")

    if SRC_PAIRS:
        print(f"\n[!] {len(SRC_PAIRS)} low-bit weight(s) with a scale tensor, e.g. "
              f"{list(SRC_PAIRS.items())[:1]}")
    norm_scales = [k for k in SRC_SHAPES if k.endswith((".scale", "_scale"))
                   and k not in SRC_AUX]
    if norm_scales:
        print(f"{len(norm_scales)} tensor(s) named *.scale are ordinary weights here "
              f"(norm scales next to full-precision weights), e.g. {norm_scales[0]}")

    orphan_q = [k for k, v in SRC_SHAPES.items()
                if v["dtype"] in LOWBIT_DTYPES - {"F8_E4M3", "F8_E5M2"}
                and len(v["shape"]) >= 2 and k not in SRC_PAIRS and k not in COMFY_LAYERS
                and k not in SRC_AUX]
    if orphan_q:
        print(f"[!] {len(orphan_q)} integer weight(s) with no scale anywhere in the file, "
              f"e.g. {orphan_q[:2]} - this cannot be converted. Get the float checkpoint.")

    # fp32 tensors inside a bf16/fp16 checkpoint are the author saying "keep these precise"
    f32 = [k for k, v in SRC_SHAPES.items() if v["dtype"] == "F32" and k not in SRC_AUX]
    f32_params = sum(int(np.prod(SRC_SHAPES[k]["shape"]) or 1) for k in f32)
    all_params = sum(int(np.prod(v["shape"]) or 1) for k, v in SRC_SHAPES.items()
                     if k not in SRC_AUX)
    SOURCE_HAS_F32_INTENT = bool(f32) and main_dtype not in ("F32", None) \
        and f32_params <= 0.15 * all_params
    if SOURCE_HAS_F32_INTENT:
        big = sorted({re.sub(r"\.\d+\.", ".N.", k) for k in f32
                      if len(SRC_SHAPES[k]["shape"]) >= 2})
        print(f"\n{len(f32)} tensor(s), {f32_params / all_params:.1%} of the parameters, are "
              f"stored F32 inside a {main_dtype} checkpoint" +
              (f" - kept F32 (KEEP_SOURCE_F32): {big[:6]}" if KEEP_SOURCE_F32 else ""))

    if SRC_BITS and SRC_BITS < 16:
        print(f"[!] {main_dtype} is {SRC_BITS} bits per weight - already quantized once.")
    hits = [h for h in PREQUANT_HINTS if h in Path(SRC_LABEL).name.lower()]
    if hits:
        print(f"[!] filename says {', '.join(hits)}")
else:
    SCALES, SRC_BITS, main_dtype = [], None, None
    SRC_PAIRS, SRC_PACKED, orphan_q = {}, [], []

## Architecture

Which families exist, how each is recognised, and which of its weights must stay F32 all
live in city96's ComfyUI-GGUF. These are read from that repo at run time rather than kept
as a copy here.

In [ ]:
# @title parse_model_templates · pull every architecture out of convert.py

def parse_model_templates(source):
    """Read the ModelXxx classes with ast. Nothing fetched is executed."""
    wanted = ("arch", "keys_detect", "keys_hiprec", "keys_banned", "keys_ignore", "shape_fix",
              "keys_noquant")
    by_class = {}

    for node in ast.walk(ast.parse(source)):
        if not isinstance(node, ast.ClassDef):
            continue
        info = {"bases": [b.id for b in node.bases if isinstance(b, ast.Name)]}
        for stmt in node.body:
            if (isinstance(stmt, ast.Assign) and len(stmt.targets) == 1
                    and isinstance(stmt.targets[0], ast.Name)
                    and stmt.targets[0].id in wanted):
                try:
                    info[stmt.targets[0].id] = ast.literal_eval(stmt.value)
                except Exception:
                    pass
        by_class[node.name] = info

    def resolve(name, key, seen=()):
        info = by_class.get(name, {})
        if key in info:
            return info[key]
        for base in info.get("bases", []):
            if base not in seen:
                got = resolve(base, key, seen + (name,))
                if got is not None:
                    return got
        return None

    # convert.py tries families in the order of its arch_list and takes the first match,
    # and that order isn't the class declaration order
    order = []
    for node in ast.parse(source).body:
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and getattr(node.targets[0], "id", None) == "arch_list"):
            order = [e.id for e in node.value.elts if isinstance(e, ast.Name)]

    out = {}
    for name in sorted(by_class, key=lambda n: order.index(n) if n in order else len(order)):
        arch = resolve(name, "arch")
        if not arch or arch == "invalid":
            continue
        out[arch] = {
            "class": name,
            "keys_detect": resolve(name, "keys_detect") or [],
            "keys_hiprec": resolve(name, "keys_hiprec") or [],
            "keys_banned": resolve(name, "keys_banned") or [],
            "keys_noquant": resolve(name, "keys_noquant") or [],
            "keys_ignore": resolve(name, "keys_ignore") or [],
            "shape_fix": bool(resolve(name, "shape_fix")),
        }
    return out

In [ ]:
# @title parse_arch_lists · pull the loader's whitelists out of loader.py

def parse_arch_lists(source):
    out = {}
    for node in ast.parse(source).body:
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and isinstance(node.targets[0], ast.Name)
                and "ARCH" in node.targets[0].id.upper()):
            try:
                out[node.targets[0].id] = set(ast.literal_eval(node.value))
            except Exception:
                pass
    return out

In [ ]:
# @title llama_quantize_archs · which names the patched llama.cpp will accept

def llama_quantize_archs():
    """city96's lcpp.patch adds a fixed set of image architectures to llama.cpp's enum.
    Anything outside it is refused with "unknown model architecture". Read from the patch
    so a name added upstream is picked up without editing this."""
    try:
        patch = requests.get(f"{RAW}/tools/lcpp.patch", timeout=60).text
        # only lines the patch adds - the diff context has llama.cpp's own archs in it
        names = set(re.findall(r'^\+\s*\{\s*LLM_ARCH_\w+,\s*"([a-z0-9_]+)"',
                               patch, re.M))
        if names:
            return names
    except Exception as e:
        print("  couldn't read lcpp.patch:", e)

    return {"flux", "sd1", "sdxl", "sd3", "aura", "ltxv", "hyvid", "wan",
            "hidream", "cosmos", "lumina2"}

In [ ]:
# @title score_families · how close is each architecture, not just which one matches

def score_families(names, templates):
    """convert.py demands every key of some keys_detect group. One renamed block in a
    merge and a checkpoint falls off a cliff from a real family to nothing at all, which
    is the moment a name gets invented and the file stops loading.

    Scoring keeps the same 1.0 bar for writing a literal name and gives everything else
    a position instead of a silence.

      1.0   every key of some group present, nothing banned - what convert.py requires
      <1.0  best group partially present: a rename, a dropped head, an added adapter
      0.0   a banned key is present, or nothing in common
    """
    names = set(names)
    ranked = []

    for arch, t in templates.items():
        banned = [k for k in t["keys_banned"] if k in names]
        if banned:
            ranked.append((arch, 0.0, f"banned key present: {banned[0]}"))
            continue

        groups = [g for g in t["keys_detect"] if g]
        if not groups:
            continue

        best, hit, total = 0.0, 0, 0
        for g in groups:
            present = sum(1 for k in g if k in names)
            if present / len(g) > best:
                best, hit, total = present / len(g), present, len(g)

        # weak corroboration: a finetune keeps the family's precision-sensitive weights
        # even when a detect key gets renamed
        corr = sum(1 for p in t["keys_hiprec"] if any(matches_any(n, [p]) for n in names))
        if corr and best < 1.0:
            best = min(0.99, best + 0.05 * corr)

        why = f"{hit}/{total} detect keys" + (f", {corr} hiprec" if corr else "")
        ranked.append((arch, round(best, 3), why))

    ranked.sort(key=lambda r: -r[1])
    return ranked


def fingerprint(shapes):
    """Structural read, independent of any template. This is what says 'a DiT shaped
    like flux' when no name fits, and it is what picks a safe stand-in."""
    names = list(shapes)
    hints = sorted({v for k, v in BLOCK_HINTS.items() if any(k in n for n in names)})

    blocks = {}
    for n in names:
        m = re.search(r"(?:^|\.)((?:[a-z_]+_)?blocks|layers)\.(\d+)\.", n)
        if m:
            blocks[m.group(1)] = max(blocks.get(m.group(1), 0), int(m.group(2)) + 1)

    dims = {}
    for n, v in shapes.items():
        if len(v["shape"]) == 2 and n.endswith(".weight"):
            dims[v["shape"][-1]] = dims.get(v["shape"][-1], 0) + 1

    conv = sum(1 for v in shapes.values() if len(v["shape"]) == 4)
    return {
        "hints": hints,
        "blocks": blocks,
        "hidden": max(dims, key=dims.get) if dims else None,
        "conv4d": conv,
        "nd5": sum(1 for v in shapes.values() if len(v["shape"]) > 4),
        "params": sum(int(np.prod(v["shape"])) for v in shapes.values()),
        "is_dit": conv < 20 and bool(blocks),
    }


def pick_quant_arch(ranked, llama_ok, templates, fp):
    """Which name llama-quantize is told. It only accepts what lcpp.patch adds, and it
    applies that family's hardcoded skip list - which restore_from_base then repairs.
    Never a shape_fix family for a DiT: those rules are aimed at 4-D conv tensors. A
    partial score from hiprec corroboration alone (no detect key at all) is not a family
    resemblance, so it doesn't pick the stand-in either: flux's rules are the plainest."""
    if STAND_IN_ARCH != "auto" and STAND_IN_ARCH in llama_ok:
        return STAND_IN_ARCH, "set by hand in 1d"
    for arch, score, why in ranked:
        if score <= 0 or arch not in llama_ok or why.startswith("0/"):
            continue
        if fp["is_dit"] and templates.get(arch, {}).get("shape_fix"):
            continue
        return arch, f"nearest family llama-quantize accepts (score {score})"
    for cand in ("flux", "lumina2", "sd3"):
        if cand in llama_ok and not templates.get(cand, {}).get("shape_fix"):
            return cand, "nothing scored - falling back to a plain DiT skip list"
    return (sorted(llama_ok)[0] if llama_ok else "flux"), "last resort"


def load_templates(repos, ref="main"):
    """Union the tables from every repo given. Earlier repos win ties, so upstream stays
    authoritative and a fork can only add families. Each repo's own tables are kept too:
    whether a file loads depends on the one loader it is opened with, not on the union."""
    merged, img, txt, seen, per_repo = {}, set(), set(), [], {}

    for repo in [r.strip() for r in re.split(r"[,\s]+", repos) if r.strip()]:
        raw = f"https://raw.githubusercontent.com/{repo}/{ref}"
        try:
            tpl = parse_model_templates(requests.get(f"{raw}/tools/convert.py",
                                                     timeout=60).text)
            lists = parse_arch_lists(requests.get(f"{raw}/loader.py", timeout=60).text)
            deq = requests.get(f"{raw}/dequant.py", timeout=60).text
        except Exception as e:
            print(f"  [!] {repo}: {e}")
            continue
        node_types = set(re.findall(r"GGMLQuantizationType\.(\w+)\s*:\s*dequantize_blocks", deq))
        per_repo[repo] = {"templates": tpl, "img": lists.get("IMG_ARCH_LIST", set()),
                          "txt": lists.get("TXT_ARCH_LIST", set()),
                          "dequant": node_types | {"F32", "F16"}}
        for arch, t in tpl.items():
            merged.setdefault(arch, dict(t, repo=repo))
        img |= per_repo[repo]["img"]
        txt |= per_repo[repo]["txt"]
        seen.append(f"{repo}@{ref} ({len(tpl)})")

    for arch, t in LOCAL_TEMPLATES.items():
        if arch not in merged:
            merged[arch] = dict(t)
            seen.append(f"built-in {arch}")

    return merged, img, txt, ", ".join(seen), per_repo


def loader_detects(repo_info, names):
    """What the loader's own detect_arch() returns for 'pig': the first family in its
    convert.py whose every key of some keys_detect group is present, unless banned."""
    names = set(names)
    for arch, t in repo_info["templates"].items():
        for group in t["keys_detect"]:
            if group and all(k in names for k in group):
                if any(k in names for k in t["keys_banned"]):
                    return None
                return arch
    return None

In [ ]:
# @title derive_hiprec · work out which weights need F32 without a template

def derive_hiprec(shapes):
    """Raw nn.Parameters - keys that don't end in .weight or .bias - are what the upstream
    keys_hiprec lists protect: modulation tables, scale-shift tables, positional embeddings,
    registers. Nothing dequantizes a bare Parameter at runtime: ComfyUI-GGUF only patches
    Linear, Conv2d, Embedding, LayerNorm and GroupNorm. A `.weight` under a real module is
    handled, so a loose pattern like "modulation" doesn't get to drag one in.

    Names come back stripped of the `model.diffusion_model.` prefix, because that is what
    every consumer matches against."""
    kept, _dropped = strip_prefix(list(shapes))
    name_of = lambda key: kept.get(key, key)
    found = set()

    for key, meta in shapes.items():
        if len(meta["shape"]) < 2 or key in SRC_AUX:
            continue
        if key.rsplit(".", 1)[-1] not in ("weight", "bias"):
            found.add(name_of(key))

    patterns = sorted({p for t in TEMPLATES.values() for p in t["keys_hiprec"]})
    patterns += [p for p in GENERIC_HIPREC if p not in patterns]
    for key in shapes:
        if key in SRC_AUX:
            continue
        name = name_of(key)
        module_weight = key.rsplit(".", 1)[-1] in ("weight", "bias")
        for p in patterns:
            if module_weight and not p.endswith((".weight", ".bias")):
                continue
            if matches_any(name, [p]):
                found.add(name)
                break

    return sorted(found), patterns

In [ ]:
print("reading architecture definitions")
TEMPLATES, IMG_ARCH, TXT_ARCH, ARCH_SOURCE, REPO_INFO = load_templates(GGUF_NODE_REPOS,
                                                                       GGUF_NODE_REF)
if not REPO_INFO:
    IMG_ARCH, TXT_ARCH = FALLBACK_IMG, FALLBACK_TXT
    ARCH_SOURCE += " (repos unreachable, built-in fallback lists)"

print(f"  {len(TEMPLATES)} families from {ARCH_SOURCE}")
for repo, info in REPO_INFO.items():
    print(f"  {repo}: loads {', '.join(sorted(info['img']))}")

DETECTED, DERIVED, ARCH_SCORES, FP = [], False, [], {}
SHIP_ARCH = QUANT_ARCH = ARCH = ARCH_NAME.strip()
NOQUANT_PATTERNS = []
LOADER = None
NODE_DEQUANT = None

if SRC_SHAPES and MODE == "diffusion":
    try:
        LLAMA_ARCH_OK = llama_quantize_archs()
    except Exception:
        LLAMA_ARCH_OK = set()

    kept, dropped = strip_prefix([k for k in SRC_SHAPES if k not in SRC_AUX])
    names = list(kept.values())
    ARCH_SCORES = score_families(names, TEMPLATES)
    FP = fingerprint(SRC_SHAPES)

    top_arch, top_score = (ARCH_SCORES[0][0], ARCH_SCORES[0][1]) if ARCH_SCORES else (None, 0.0)
    confident = top_score >= CONFIDENT_AT
    DETECTED = [a for a, s, _ in ARCH_SCORES if s >= CONFIDENT_AT]

    # the loader the files are meant for
    repos = list(REPO_INFO)
    if LOAD_WITH != "auto" and LOAD_WITH.strip():
        LOADER = LOAD_WITH.strip()
        if LOADER not in REPO_INFO:
            print(f"  reading {LOADER} too, since 1d says the files will be loaded with it")
            _t, _i, _x, _s, extra = load_templates(LOADER, GGUF_NODE_REF)
            REPO_INFO.update(extra)
    else:
        want = ARCH_NAME.strip() if ARCH_MODE == "force" and ARCH_NAME.strip() else top_arch
        LOADER = next((r for r in repos if want and want in REPO_INFO[r]["img"]), None) \
            or next((r for r in repos if loader_detects(REPO_INFO[r], names)), None) \
            or (repos[0] if repos else None)
    info = REPO_INFO.get(LOADER, {"img": IMG_ARCH, "templates": TEMPLATES})
    NODE_DEQUANT = REPO_INFO.get(LOADER, {}).get("dequant")
    pig_arch = loader_detects(info, names) if LOADER in REPO_INFO else None

    if ARCH_MODE == "force" and ARCH_NAME.strip():
        SHIP_ARCH = QUANT_ARCH = ARCH_NAME.strip()
        VERDICT = "forced by hand in 1d"
    elif confident and top_arch in info["img"]:
        SHIP_ARCH = QUANT_ARCH = top_arch
        VERDICT = f"exact match, and {LOADER} loads it"
    elif pig_arch and DEFER_WHEN_UNSURE:
        SHIP_ARCH, QUANT_ARCH = "pig", pig_arch
        VERDICT = f"{LOADER}'s own detection recognises it as {pig_arch!r} - shipping as 'pig'"
    elif confident:
        SHIP_ARCH = QUANT_ARCH = top_arch
        who = [r for r, i in REPO_INFO.items() if top_arch in i["img"]]
        VERDICT = (f"exact match {top_arch!r}, but {LOADER} doesn't load it" +
                   (f" - load with {who[0]} (set LOAD_WITH in 1d)" if who else
                    " - no listed loader does; add a fork that knows it to 1d"))
    else:
        SHIP_ARCH = QUANT_ARCH = (ARCH_NAME.strip() or
            re.sub(r"[^a-z0-9]+", "_", Path(SRC_LABEL).stem.lower()).strip("_")[:24])
        VERDICT = ("no family matches these weights and no listed loader recognises them - "
                   "the file is converted, but no GGUF node you listed will load it")

    ARCH = QUANT_ARCH
    DERIVED = not DETECTED and not (ARCH_MODE == "force" and ARCH_NAME.strip())

    print(f"\nstructure: {FP['params'] / 1e9:.2f}B params, hidden {FP['hidden']}, "
          f"{FP['conv4d']} 4-D, {FP['nd5']} 5-D, "
          f"{'DiT' if FP['is_dit'] else 'conv unet'}")
    if FP["blocks"]:
        print("  blocks    : " + ", ".join(f"{k}={v}" for k, v in FP["blocks"].items()))
    if FP["hints"]:
        print("  looks like: " + "; ".join(FP["hints"]))
    if dropped:
        print(f"  [!] the prefix rule drops {len(dropped)}: {dropped[:3]}")

    print("\nfamily scores:")
    for arch, score, why in ARCH_SCORES[:5]:
        mark = "  <- picked" if arch == top_arch and confident else ""
        load = "" if arch in IMG_ARCH else "  (no listed loader has it)"
        src_repo = TEMPLATES.get(arch, {}).get("repo", "")
        print(f"  {score:>5.2f}  {arch:<14} {why}  [{src_repo}]{load}{mark}")

    print(f"\n{VERDICT}")
    print(f"  load with             : {LOADER}")
    print(f"  convert as            : {QUANT_ARCH}")
    print(f"  ship as               : {SHIP_ARCH}")

if MODE == "diffusion" and SRC_SHAPES:
    tpl = TEMPLATES.get(ARCH, {})
    NOQUANT_PATTERNS = list(tpl.get("keys_noquant", []))
    if F32_OVERRIDE is not None:
        F32_PATTERNS = F32_OVERRIDE
        print("\nkeep in F32 (set in 1e):", ", ".join(F32_PATTERNS) or "nothing")
    else:
        from_template = list(tpl.get("keys_hiprec", []))
        derived, _pats = derive_hiprec(SRC_SHAPES)
        extra = [p for p in derived if not matches_any(p, from_template)]
        F32_PATTERNS = from_template + extra
        print(f"\nkeep in F32: {len(F32_PATTERNS)} pattern(s) - {len(from_template)} from "
              f"the {ARCH} template, {len(extra)} derived from these tensor names")
        if extra:
            print("  derived:", ", ".join(extra[:6]),
                  f"(+{len(extra) - 6} more)" if len(extra) > 6 else "")
    if NOQUANT_PATTERNS:
        print(f"keep at source precision: {NOQUANT_PATTERNS}")

### Survey a folder (optional)

Headers only, so a folder of forty checkpoints takes seconds. Needs 1a-1e and the architecture cell above to have run. For survey only, set `JOB` to `check only` in 1a so nothing tries to download.

In [ ]:
# @title 1g · Survey a folder before converting any of it { display-mode: "form" }

# @markdown Headers only, so forty checkpoints take seconds. One row each: which family
# @markdown it scores as, what it would ship as, and whether its weights are already
# @markdown quantized. Leave the path empty to skip.
SURVEY_DIR = ""  # @param {type:"string"}

if SURVEY_DIR.strip():
    folder = Path(SURVEY_DIR).expanduser().resolve()
    files = sorted(folder.rglob("*.safetensors")) if folder.is_dir() else [folder]
    print(f"{len(files)} file(s) under {folder}\n")

    head = f"{'file':<40} {'best':<12} {'conf':>5}  {'ships as':<9} note"
    print(head); print("-" * len(head))

    for f in files:
        try:
            shapes = read_header(f)
        except Exception as e:
            print(f"{f.name[:40]:<40} {'-':<12} {'-':>5}  {'-':<9} unreadable: {e}")
            continue

        kept, _ = strip_prefix(list(shapes))
        ranked = score_families(kept.values(), TEMPLATES)
        arch, conf = (ranked[0][0], ranked[0][1]) if ranked else ("none", 0.0)

        pairs, packed = find_scale_pairs(shapes)
        comfyq = sum(1 for k in shapes if k.endswith(".comfy_quant"))
        sub16 = [k for k, v in shapes.items()
                 if v["dtype"] in LOWBIT_DTYPES and len(v["shape"]) >= 2]

        if conf >= CONFIDENT_AT and arch in IMG_ARCH:
            ships, note = arch, "clean"
        elif conf >= CONFIDENT_AT:
            ships, note = arch, "no loader on your list has this family"
        else:
            ships, note = "pig", "partial - detect_arch() decides at load"

        if comfyq:
            note += f"; ComfyUI-quantized ({comfyq} layers) - decoded via ComfyUI"
        elif packed:
            note += f"; bit-packed ({packed[0]}), cannot convert"
        elif sub16 and not pairs:
            note += f"; {len(sub16)} sub-16-bit weights, NO scales in the file"
        elif sub16:
            note += f"; {len(sub16)} sub-16-bit, {len(pairs)} scale pairs"

        print(f"{f.name[:40]:<40} {arch:<12} {conf:>5.2f}  {ships:<9} {note}")

    print("\nRows saying 'pig' still convert - they just let the node identify them.")
    print("Rows saying NO scales are the ones that come out as noise. Use the float")
    print("checkpoint for those.")
else:
    print("SURVEY_DIR empty, skipping")

## What can actually be built

An already-quantized checkpoint doesn't rule conversion out, it just limits it. This works
out what's possible before any time is spent building a toolchain, and drops the targets
that can't come out better than the source rather than failing partway through.

In [ ]:
# @title bits_of · how many bits per weight a precision label means

def bits_of(label):
    if label == "mix":
        return None
    if label.upper() in ("F16", "BF16"):
        return 16

    m = re.search(r"I?Q(\d)", label.upper())
    return int(m.group(1)) if m else None

In [ ]:
# @title apply_scale · put a weight back on its original scale

def apply_scale(weight, scale, inverse=False):
    import torch

    weight = weight.to(torch.float32)
    scale = scale.to(torch.float32)
    if inverse:
        scale = 1.0 / scale

    if scale.numel() == 1:
        return weight * scale.reshape(())
    try:
        return weight * scale                      # already broadcastable
    except RuntimeError:
        pass
    if scale.numel() == weight.shape[0]:           # one per output channel
        return weight * scale.reshape(-1, *([1] * (weight.ndim - 1)))
    if scale.numel() == weight.shape[-1]:          # one per input channel
        return weight * scale.reshape(*([1] * (weight.ndim - 1)), -1)

    raise ValueError(f"scale {tuple(scale.shape)} doesn't line up with weight "
                     f"{tuple(weight.shape)}")

In [ ]:
# @title comfy_runtime · ComfyUI itself, for decoding its quant formats and the final check

COMFY_READY = False

def ensure_comfy():
    """Clones ComfyUI and imports it CPU-only. Used to decode ComfyUI-quantized layers
    exactly the way ComfyUI does, and by the final load check."""
    global COMFY_READY
    if COMFY_READY:
        return
    if not (COMFY / "comfy").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", COMFYUI_REF.strip() or "master",
                        "https://github.com/comfyanonymous/ComfyUI", str(COMFY)], check=True)
        # only what `import comfy.sd` needs, pinned the way ComfyUI pins them
        reqs = (COMFY / "requirements.txt").read_text().splitlines()
        wanted = ("comfy-kitchen", "comfy-aimdo", "torchsde", "einops", "simpleeval",
                  "blake3", "sentencepiece", "tokenizers", "transformers", "safetensors",
                  "torchvision", "aiohttp", "pyyaml", "scipy", "psutil", "pydantic")
        pins = [r.split("#")[0].strip() for r in reqs
                if r.split("#")[0].strip() and re.split(r"[<>=~ ]", r.strip())[0] in wanted]
        missing = []
        for p in pins:
            mod = re.split(r"[<>=~ ]", p)[0].replace("-", "_")
            if mod in ("torchvision", "transformers", "scipy", "psutil", "pyyaml",
                       "safetensors", "aiohttp", "tokenizers", "sentencepiece", "pydantic"):
                try:
                    __import__({"pyyaml": "yaml"}.get(mod, mod))
                    continue                       # keep Colab's own build of these
                except ImportError:
                    pass
            missing.append(p)
        if missing:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    if str(COMFY) not in sys.path:
        sys.path.insert(0, str(COMFY))
    argv = sys.argv
    sys.argv = ["comfy", "--cpu"]
    try:
        import comfy.options
        comfy.options.enable_args_parsing(True)
        import comfy.ops, comfy.quant_ops, comfy.sd       # noqa: F401
    finally:
        sys.argv = argv
    COMFY_READY = True


# logical K = stored columns x this, for formats that pack several values per byte
COMFY_PACK = {"nvfp4": 2, "convrot_w4a4": 2, "asym_w4a8_int8": 2, "w6a8_int8": 4 / 3}


def comfy_dequant(tensors, weight_key, conf):
    """One ComfyUI-quantized layer back to float32, through the same layout classes ComfyUI
    uses at inference. `tensors` maps every key of that layer to its tensor."""
    ensure_comfy()
    import comfy.ops, comfy.quant_ops as cq
    fmt = conf.get("format")
    if fmt not in cq.QUANT_ALGOS:
        raise ValueError(f"format {fmt!r} isn't one this ComfyUI knows - update COMFYUI_REF")
    algo = cq.QUANT_ALGOS[fmt]
    prefix = weight_key[: -len("weight")]
    w = tensors[weight_key]
    if w.ndim != 2:
        raise ValueError(f"{weight_key}: {w.ndim}-D quantized weight, only Linear is supported")
    st = algo["storage_t"]
    if w.dtype == torch.uint8 and st != torch.uint8 and st.itemsize == 1:
        w = w.view(st)                     # fp8 written to disk as raw bytes
    n, kp = w.shape
    k = int(round(kp * COMFY_PACK.get(fmt, 1)))
    ops = comfy.ops.mixed_precision_ops({}, compute_dtype=torch.float32, full_precision_mm=True)
    lin = ops.Linear(k, n, bias=False)
    sd = {kk[len(prefix):]: v for kk, v in tensors.items()
          if kk.startswith(prefix) and kk != prefix + "bias"}
    sd["weight"] = w
    lin.load_state_dict(sd, strict=False)
    out = lin.weight
    if isinstance(out, cq.QuantizedTensor):
        out = out.dequantize()
    out = out.detach().float()
    if tuple(out.shape) != (n, k):
        raise ValueError(f"{weight_key}: decoded to {tuple(out.shape)}, expected {(n, k)}")
    return out


class SourceReader:
    """Every source tensor the way the model really holds it: ComfyUI-quantized layers
    decoded by ComfyUI, scale pairs multiplied out, everything else as stored. The
    conversion and every value check read through this, so a check can never agree with a
    wrong reconstruction just because it made the same mistake."""

    def __init__(self):
        from safetensors import safe_open
        self._open = safe_open
        self.handles = {}

    def raw(self, key):
        path = SRC_INDEX[key]
        if path not in self.handles:
            self.handles[path] = self._open(str(path), framework="pt", device="cpu")
        return self.handles[path].get_tensor(key)

    def get(self, key):
        if key in COMFY_LAYERS and UNPACK:
            stem = key[: -len("weight")]
            layer = {k: self.raw(k) for k in SRC_SHAPES
                     if k.startswith(stem) and "." not in k[len(stem):]}
            return comfy_dequant(layer, key, COMFY_LAYERS[key])
        t = self.raw(key)
        if key in SRC_PAIRS and UNPACK:
            t = apply_scale(t, self.raw(SRC_PAIRS[key]), inverse=SCALE_INVERSE)
        return t

    def close(self):
        self.handles.clear()

    def __enter__(self):
        return self

    def __exit__(self, *a):
        self.close()

In [ ]:
# @title scale_direction · multiply or divide, for scale pairs with no metadata

def scale_direction(pairs, sample=8):
    """Only for scale pairs that carry no ComfyUI metadata. `weight_scale_inv` is named for
    the reciprocal but DeepSeek-style checkpoints dequantize with `weight * scale_inv`, so the
    direction is settled by trying both and keeping whichever lands in a plausible range.
    Legacy ComfyUI `.scale_weight` is always a multiply."""
    if not pairs:
        return False, "no pairs"
    if all(s.endswith(".scale_weight") for s in pairs.values()):
        return False, "weight * scale (ComfyUI scaled_fp8)"

    picked = list(pairs.items())[:sample]
    scores = {}
    with SourceReader() as r:
        for inverse in (False, True):
            stds = []
            for weight, scale in picked:
                try:
                    out = apply_scale(r.raw(weight), r.raw(scale), inverse=inverse)
                    stds.append(float(out.to(torch.float32).std()))
                except Exception:
                    stds.append(float("nan"))
            good = sum(1 for s in stds if SANE_STD[0] <= s <= SANE_STD[1])
            scores[inverse] = (good, stds)

    fwd, inv = scores[False][0], scores[True][0]
    if fwd == inv == 0:
        return False, ("neither direction gives sane weights - the scales may be blockwise, or "
                       "this needs the original quantization library")
    inverse = inv > fwd
    op = "weight / scale" if inverse else "weight * scale"
    return inverse, f"{op} ({max(fwd, inv)}/{len(picked)} sampled land in range)"


def precheck_scale_shapes(pairs):
    """Every pair has to broadcast, checked from the headers before a byte is converted -
    finding a blockwise scale half way through the file is too late."""
    bad = []
    for w, s in pairs.items():
        ws, ss = SRC_SHAPES[w]["shape"], SRC_SHAPES[s]["shape"]
        n = int(np.prod(ss)) if ss else 1
        if n == 1 or list(ss) == list(ws):
            continue
        if ws and n == ws[0] and (len(ss) <= 1 or ss[-1] == 1 or len(ss) == len(ws)):
            continue
        if ws and n == ws[-1]:
            continue
        bad.append(f"{w} {ws} with scale {ss}")
    return bad

In [ ]:
# @title plan_targets · work out which of the requested files are worth building

def plan_targets():
    """Nothing to weigh up if the source is 16-bit or better.

    Below that, the source only holds N bits of information per weight no matter how it gets
    stored. Writing it as BF16 doubles the file and adds nothing - which is exactly what an
    earlier version of this notebook did, turning a 12 GB int8 checkpoint into 24 GB. The
    size-matched landing spot for an 8-bit source is Q8_0, not a float type.
    """
    plan = []

    for label in EXPECT:
        want = None if label == "mix" else bits_of(label)

        if not SRC_BITS or SRC_BITS >= 16:
            plan.append((label, "build", ""))

        elif want is None:
            if BASE_QTYPE:
                plan.append((label, "build",
                             f"written as {BASE_QTYPE} - same size as the source, holds the "
                             f"unpacked values. Storing {SRC_BITS}-bit data as a float type "
                             f"would double the file for nothing"))
            else:
                plan.append((label, "too high",
                             f"a float copy of {SRC_BITS}-bit data is twice the size and no "
                             f"better"))

        elif want > SRC_BITS:
            plan.append((label, "too high",
                         f"{want}-bit from a {SRC_BITS}-bit source is about "
                         f"{want // SRC_BITS}x the size for the same quality"))

        elif want == SRC_BITS:
            plan.append((label, "build",
                         f"the size-matched target for a {SRC_BITS}-bit source"))

        else:
            plan.append((label, "build",
                         f"{want}-bit from a {SRC_BITS}-bit source stacks two rounding steps, "
                         f"but it is a real size saving"))

    return plan

In [ ]:
# @title python_quant_types · formats gguf-py can write AND the GGUF node can read

def python_quant_types():
    """gguf-py can produce types no GGUF node dequantizes (MXFP4, TQ1_0, TQ2_0). A tensor in
    one of those is either a load error or packed bytes used as a weight. Only types the
    loader from 1d implements are offered."""
    probe = np.random.default_rng(0).standard_normal((32, 256)).astype(np.float32)
    node = NODE_DEQUANT or {"BF16", "Q8_0", "Q5_1", "Q5_0", "Q4_1", "Q4_0"}
    usable = {}
    for t in gguf.GGMLQuantizationType:
        if t.name in ("F32", "F16") or t.name not in node:
            continue
        try:
            gguf.quants.quantize(probe, t)
            usable[t.name] = t
        except Exception:
            pass
    return usable

In [ ]:
# @title already_built · is this target already sitting in the output folder

def output_tails(label):
    """The suffix a target lands under. `mix` is whichever base type the checkpoint
    resolves to, worked out the same way convert_diffusion works it out, so which file to
    look for is known before anything has been converted."""
    if label != "mix":
        return [label.upper()]
    if BASE_QTYPE:
        return [BASE_QTYPE.upper()]
    return ["BF16", "F16"]


def already_built(label):
    """A file only counts as built when all three hold: it's named the way the converter
    would have named it, it's the full length its own header describes, and its stamp is
    the checkpoint from 1b. A name on its own is not evidence - the same filename gets
    reused across versions of a model, and reusing the wrong one is invisible until the
    output is noise."""
    if SRC is None:
        return None
    stem = SRC.name if SRC.is_dir() else SRC.stem
    tails = {t.upper() for t in output_tails(label)}

    for path in sorted(OUT.glob("*.gguf")):
        if path.stat().st_size == 0 or path.name.startswith("mmproj"):
            continue
        head, _, tail = path.stem.rpartition("-")
        if tail.upper() not in tails or head != stem:
            continue

        ok, note = gguf_is_complete(path)
        if not ok:
            print(f"  {path.name}: {note} - an interrupted write, building it again")
            continue

        verdict, detail = stamp_matches(path)
        if verdict == "different":
            print(f"  {path.name} was made from a different checkpoint ({detail}) - "
                  f"building it again")
            continue
        if verdict == "unstamped":
            print(f"  {path.name} has no source stamp, so there's no way to tell what it "
                  f"was made from - building it again")
            continue

        return path
    return None


def has_over_4d(path):
    """restore_nd writes the stashed tensors back with all their dimensions. llama-quantize
    can't read anything over 4, so a base file that has already been through that pass can
    be shipped but can't be quantized from."""
    reader = gguf.GGUFReader(str(path))
    over = any(len([d for d in t.shape if int(d)]) > 4 for t in reader.tensors)
    del reader
    return over

In [ ]:
UNPACK = False
BLOCKED = None
SCALE_INVERSE = False

if SRC_PACKED and not COMFY_LAYERS:
    BLOCKED = (f"bit-packed weights ({SRC_PACKED[:2]}) need the original quantization "
               f"library to reverse")
elif (COMFY_LAYERS or SRC_PAIRS) and MODE == "diffusion":
    print(f"already quantized: {len(COMFY_LAYERS)} ComfyUI layer(s), "
          f"{len(SRC_PAIRS)} other scale pair(s)")
    if IF_ALREADY_QUANTIZED == "stop":
        BLOCKED = "IF_ALREADY_QUANTIZED is 'stop'"
    elif IF_ALREADY_QUANTIZED == "unpack and convert":
        UNPACK = True
        if SRC_PAIRS:
            bad = precheck_scale_shapes(SRC_PAIRS)
            if bad:
                BLOCKED = (f"{len(bad)} scale(s) don't broadcast onto their weight (blockwise "
                           f"or packed layout with no ComfyUI metadata), e.g. {bad[0]}")
            else:
                SCALE_INVERSE, note = scale_direction(SRC_PAIRS)
                print(f"  scale pairs: {note}")
                if note.startswith("neither"):
                    BLOCKED = "the scales don't reconstruct sane weights in either direction"
        if COMFY_LAYERS and not BLOCKED:
            # decode one layer of every format now: a format this ComfyUI can't reverse
            # stops the run here, not after an hour of converting
            print("  loading ComfyUI to decode its quantization formats")
            ensure_comfy()
            seen = {}
            for w, conf in COMFY_LAYERS.items():
                tag = json.dumps(conf, sort_keys=True)
                seen.setdefault(tag, w)
            with SourceReader() as r:
                for tag, w in seen.items():
                    try:
                        t = r.get(w)
                        s = float(t.std())
                        ok = np.isfinite(s) and SANE_STD[0] / 10 <= s <= SANE_STD[1] * 10
                        print(f"    {json.loads(tag).get('format')}"
                              f"{' +convrot' if json.loads(tag).get('convrot') else ''}: "
                              f"{w} decodes to std {s:.4g}" + ("" if ok else "  [!] implausible"))
                        if not ok:
                            BLOCKED = f"{w} decodes to implausible values (std {s:.3g})"
                    except Exception as e:
                        BLOCKED = (f"ComfyUI can't decode {json.loads(tag).get('format')!r} "
                                   f"({type(e).__name__}: {e}). Update COMFYUI_REF, or use "
                                   f"the bf16 checkpoint.")
                        break
    else:
        print("  [!] converting as-is: the packed integers will be written as the weights "
              "and the output will be noise")

BASE_QTYPE = None
if not BLOCKED and SRC_BITS and SRC_BITS <= 8 and MODE == "diffusion":
    BASE_QTYPE = "Q8_0"
    print(f"  source is {SRC_BITS}-bit, so the base file will be Q8_0 "
          f"(about the same size) instead of a float type (about double)")

if BLOCKED:
    print(f"\ncannot convert: {BLOCKED}")
    print("Use the full-precision checkpoint, or set IF_ALREADY_QUANTIZED in 1d.")
    DO_CONVERT = False
else:
    PLAN = plan_targets()
    print("\nplan:")
    for label, verdict, why in PLAN:
        mark = "build" if verdict == "build" else IF_TARGET_TOO_HIGH
        print(f"  {label:10} {mark:16} {why}")

    if any(v == "too high" for _, v, _ in PLAN) and IF_TARGET_TOO_HIGH == "stop":
        raise RuntimeError("some targets can't improve on the source and "
                           "IF_TARGET_TOO_HIGH is 'stop'")

    if IF_TARGET_TOO_HIGH == "drop it":
        keep = [l for l, v, _ in PLAN if v == "build"]
        if BASE_QTYPE and "mix" in keep and BASE_QTYPE in keep:
            keep.remove(BASE_QTYPE)
            print(f"  {BASE_QTYPE} and mix are the same file here")
        dropped = [l for l, v, _ in PLAN if v != "build"]
        if dropped:
            print("\ndropping:", ", ".join(dropped))
        EXPECT = keep
        QUANTS = [q for q in QUANTS if q in keep]
        WANT_MIX = "mix" in keep

    if not EXPECT:
        print("\nnothing left worth building from this checkpoint.")
        DO_CONVERT = False

REUSE_OK = IF_OUTPUT_EXISTS == "skip"
REUSE = {}
if DO_CONVERT and REUSE_OK:
    for label in EXPECT:
        found = already_built(label)
        if found:
            REUSE[label] = found
    if REUSE:
        print(f"\nalready in {OUT}:")
        for label, path in REUSE.items():
            print(f"  {label:10} {path.name}  ({path.stat().st_size / 2**30:.2f} GB)")
        QUANTS = [q for q in QUANTS if q not in REUSE]
        WANT_MIX = WANT_MIX and "mix" not in REUSE
        if all(label in REUSE for label in EXPECT):
            print("  everything asked for is there - nothing to convert")
            DO_CONVERT = False

TODO = [label for label in EXPECT if label not in REUSE]
NEED_LLAMA = bool(QUANTS) or MODE == "llm"
print(f"\nbuilding: {', '.join(TODO) or 'nothing'}"
      + (f"  (keeping {', '.join(REUSE)})" if REUSE else ""))

QUANT_ENGINE = "llama"
REQUANTIZE = bool(BASE_QTYPE)
STAND_IN = None
PY_TYPES = {}
if DO_CONVERT and MODE == "diffusion" and QUANTS:
    LLAMA_ARCH_OK = llama_quantize_archs()
    PY_TYPES = python_quant_types()
    if REQUANTIZE:
        print(f"\nthe base file is {BASE_QTYPE}, so anything further is a requantize")

    def substitute(cannot, available, why):
        global EXPECT, QUANTS
        print(f"  {why}: {', '.join(cannot)}")
        swaps = {q: NEAREST_QUANT.get(q) for q in cannot}
        if IF_QUANT_UNAVAILABLE == "substitute the nearest":
            ok = {q: s for q, s in swaps.items() if s in available}
            if ok:
                print(f"  substituting {', '.join(f'{a} -> {b}' for a, b in ok.items())}")
            EXPECT = list(dict.fromkeys(ok.get(e, e) for e in EXPECT
                                        if e not in cannot or e in ok))
            QUANTS = list(dict.fromkeys(ok.get(q, q) for q in QUANTS
                                        if q not in cannot or q in ok))
        else:
            EXPECT = [e for e in EXPECT if e not in cannot]
            QUANTS = [q for q in QUANTS if q not in cannot]

    rejects = [q for q in QUANTS if q in LLAMA_IMG_REJECTS]
    if rejects:
        substitute(rejects, set(QUANTS) | set(PY_TYPES) | {"Q4_K_M", "Q4_K_S", "Q4_1", "Q4_0"},
                   "llama-quantize refuses IQ types for image models")

    if ARCH not in LLAMA_ARCH_OK:
        print(f"\nllama-quantize doesn't know {ARCH!r}. It only accepts "
              f"{', '.join(sorted(LLAMA_ARCH_OK))}.")
        if IF_ARCH_UNKNOWN == "drop the quants":
            print("  dropping", ", ".join(QUANTS))
            EXPECT = [e for e in EXPECT if e not in QUANTS]
            QUANTS = []
        elif IF_ARCH_UNKNOWN == "stand-in arch":
            STAND_IN, why = pick_quant_arch(ARCH_SCORES, LLAMA_ARCH_OK, TEMPLATES, FP)
            print(f"  quantizing under {STAND_IN!r} ({why}), then putting {ARCH!r} back.")
            print(f"  Every tensor the base file keeps F32 is copied back after each quant, so")
            print(f"  {STAND_IN!r}'s tensor rules can't touch what this notebook protects.")
        else:
            QUANT_ENGINE = "python"
            cannot = [q for q in QUANTS if q not in PY_TYPES]
            if cannot:
                substitute(cannot, set(PY_TYPES), "gguf-py can't build these (K-quants are "
                                                  "C++ only)")
            print(f"  quantizing in python: {', '.join(QUANTS) or 'nothing'}")

    if NODE_DEQUANT:
        unreadable = [q for q in QUANTS if not ({"Q4_K_M": "Q4_K", "Q4_K_S": "Q4_K",
                      "Q5_K_M": "Q5_K", "Q5_K_S": "Q5_K", "Q3_K_M": "Q3_K", "Q3_K_S": "Q3_K",
                      "Q3_K_L": "Q3_K"}.get(q, q) in NODE_DEQUANT)]
        if unreadable:
            print(f"  [!] {LOADER} can't dequantize {unreadable} - dropping them")
            EXPECT = [e for e in EXPECT if e not in unreadable]
            QUANTS = [q for q in QUANTS if q not in unreadable]

    NEED_LLAMA = bool(QUANTS) and QUANT_ENGINE == "llama"

## Toolchain

In [ ]:
def build_quantize(src_dir):
    !cd {str(src_dir)} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null 2>&1
    !cd {str(src_dir)} && cmake --build build --config Release -j --target llama-quantize > /dev/null 2>&1
    return src_dir / "build" / "bin" / "llama-quantize"

if DO_CONVERT:
    if not GG.exists():
        !git clone -q --depth 1 -b {GGUF_NODE_REF.strip() or 'main'} https://github.com/{GGUF_NODE_REPO.strip()} {str(GG)}
    print("ComfyUI-GGUF:", GG)

    if MODE == "diffusion" and NEED_LLAMA:
        if not (LCPP / "build" / "bin" / "llama-quantize").exists():
            if not LCPP.exists():
                !git clone -q --depth 1 -b {LLAMA_TAG.strip()} https://github.com/ggerganov/llama.cpp {str(LCPP)}
                patch = (GG / "tools" / "lcpp.patch")
                patch.write_bytes(patch.read_bytes().replace(b"\r\n", b"\n"))
                !cd {str(LCPP)} && git apply {str(patch)}
            # an unapplied patch still builds - into a llama-quantize that rejects every
            # image architecture. Check for it rather than trusting the exit code.
            if "LLM_ARCH_FLUX" not in (LCPP / "src" / "llama.cpp").read_text(errors="ignore"):
                raise RuntimeError(f"lcpp.patch did not apply to llama.cpp {LLAMA_TAG}. Delete "
                                   f"{LCPP} and check LLAMA_TAG against {GGUF_NODE_REPO}'s README.")
            print("building patched llama-quantize, this takes a few minutes")
            build_quantize(LCPP)
        QUANTIZE = LCPP / "build" / "bin" / "llama-quantize"

    elif MODE == "llm":
        # LLM conversion needs a llama.cpp that knows the model; the image patch pins 2024
        if not LCPP_LLM.exists():
            !git clone -q --depth 1 -b {LLM_LLAMA_REF.strip() or 'master'} https://github.com/ggerganov/llama.cpp {str(LCPP_LLM)}
            !pip install -q "transformers>=4.45" sentencepiece protobuf mistral-common
        if NEED_LLAMA and not (LCPP_LLM / "build" / "bin" / "llama-quantize").exists():
            print("building llama-quantize, this takes a few minutes")
            build_quantize(LCPP_LLM)
        QUANTIZE = LCPP_LLM / "build" / "bin" / "llama-quantize"
    else:
        QUANTIZE = LCPP / "build" / "bin" / "llama-quantize"

    print("llama-quantize:", "ready" if QUANTIZE.exists() else "not built (not needed)")
    if NEED_LLAMA and not QUANTIZE.exists():
        raise RuntimeError("llama-quantize failed to build - rerun with the cmake output visible")
else:
    QUANTIZE = None
    print("check only, skipping the toolchain")

## Convert

In [ ]:
# @title convert_diffusion · one tensor at a time, so RAM never holds the whole model

F32_T = gguf.GGMLQuantizationType.F32
F16_T = gguf.GGMLQuantizationType.F16
BF16_T = gguf.GGMLQuantizationType.BF16


def pick_base_type(name, shape, src_dtype, main):
    """The one place that decides what a tensor is stored as in the base file. Quants are
    cut from the base file, and every tensor it holds as F32 is restored after quantizing,
    so F32 here means F32 in every file.

    Returns (qtype, why)."""
    n_params = int(np.prod(shape)) if shape else 1
    ndim = len(shape)
    leaf = name.rsplit(".", 1)[-1]

    # one row: [dim] or [1, dim]. Upstream's `n_dims == 1` misses [1, dim] raw Parameters
    if int(np.prod(shape[:-1] or [1])) == 1:
        return F32_T, "1-D" if ndim <= 1 else f"single row {list(shape)}"
    if n_params <= 1024:
        return F32_T, "small"
    if F32_PATTERNS and matches_any(name, F32_PATTERNS):
        return F32_T, "protected"
    # the GGUF node only dequantizes Linear, Conv2d, Embedding, LayerNorm and GroupNorm.
    # It counts BF16 as quantized, so a BF16 raw Parameter, Conv1d or Conv3d weight arrives
    # as packed bytes. Those have to be a type torch reads directly.
    if leaf not in ("weight", "bias") or ndim not in (2, 4):
        return F32_T, f"no GGUF layer reads a {ndim}-D {leaf!r}"
    if KEEP_SOURCE_F32 and SOURCE_HAS_F32_INTENT and src_dtype == "F32":
        return F32_T, "F32 in the source"
    if NOQUANT_PATTERNS and matches_any(name, NOQUANT_PATTERNS):
        return (BF16_T if src_dtype in ("BF16", "F32") else F16_T), "keep source precision"
    if BASE_QTYPE:
        return getattr(gguf.GGMLQuantizationType, BASE_QTYPE), "base type"
    if src_dtype == "BF16" or (UNPACK and src_dtype in LOWBIT_DTYPES and main == "BF16"):
        return BF16_T, ""
    if src_dtype == "F32" and main in ("BF16", None):
        return BF16_T, ""
    return F16_T, ""


def convert_diffusion(source, dst_pattern):
    """Reads one tensor at a time and lets the writer spill to disk, so peak RAM is a few
    times the largest single tensor instead of twice the model."""
    if str(GG / "tools") not in sys.path:
        sys.path.insert(0, str(GG / "tools"))
    import convert as base   # city96 tools/convert.py
    template = {t.arch: t for t in base.arch_list}.get(ARCH)
    shape_fix = bool(getattr(template, "shape_fix", False)) and not DERIVED
    ignore = list(getattr(template, "keys_ignore", []) or []) + \
        list(TEMPLATES.get(ARCH, {}).get("keys_ignore", []) or [])

    keys = [k for k in SRC_SHAPES if k not in SRC_AUX or not UNPACK]
    if not UNPACK:
        keys = [k for k in keys if not k.endswith(".comfy_quant")]
    kept, dropped = strip_prefix(keys)
    if dropped:
        print(f"  prefix rule drops {len(dropped)} tensor(s)")

    counts = {}
    for k in kept:
        if len(SRC_SHAPES[k]["shape"]) >= 2:
            d = SRC_SHAPES[k]["dtype"]
            counts[d] = counts.get(d, 0) + 1
    main = max(counts, key=counts.get) if counts else "F16"
    if UNPACK and main in LOWBIT_DTYPES:
        # what the unpacked weights are written as: the precision the rest of the file uses
        floats = {d: c for d, c in counts.items() if d in ("BF16", "F16", "F32")}
        main = max(floats, key=floats.get) if floats else "BF16"

    if BASE_QTYPE:
        ftype_name = BASE_QTYPE
        ftype = getattr(gguf.LlamaFileType, f"MOSTLY_{BASE_QTYPE}", gguf.LlamaFileType.MOSTLY_Q8_0)
    elif main == "F16":
        ftype_name, ftype = "F16", gguf.LlamaFileType.MOSTLY_F16
    else:
        ftype_name, ftype = "BF16", gguf.LlamaFileType.MOSTLY_BF16

    dst = Path(str(dst_pattern).replace("{ftype}", ftype_name)).resolve()
    dst.parent.mkdir(parents=True, exist_ok=True)

    writer = gguf.GGUFWriter(path=None, arch=ARCH, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    write_stamp(writer)

    todo = [k for k in kept if not any(x in kept[k] for x in ignore)]
    width = min(48, max((len(kept[k]) for k in todo), default=20))
    bar = None
    if PROGRESS == "summary":
        from tqdm.auto import tqdm
        bar = tqdm(total=len(todo), unit="tensor")

    held, tally, written, reasons = {}, {}, 0, {}
    by_file = {}
    for k in todo:
        by_file.setdefault(SRC_INDEX[k], []).append(k)

    with SourceReader() as reader:
        for path, file_keys in by_file.items():
            for key in file_keys:
                name = kept[key]
                src_dtype = SRC_SHAPES[key]["dtype"]
                data = reader.get(key) if UNPACK else reader.raw(key)
                if data.dtype in (getattr(torch, "float8_e4m3fn", None),
                                  getattr(torch, "float8_e5m2", None)):
                    data = data.to(torch.float16)
                if not data.is_floating_point():
                    data = data.to(torch.float32)
                arr = data.to(torch.float32).numpy() if data.dtype != torch.float32 \
                    else data.numpy()
                del data

                if arr.ndim > 4:
                    held[name] = torch.from_numpy(arr.copy())
                    if bar:
                        bar.update(1)
                    del arr
                    continue

                qtype, why = pick_base_type(name, list(arr.shape), src_dtype, main)
                # F16 can't hold what BF16/F32 can: anything that would overflow goes BF16
                if qtype == F16_T and np.abs(arr).max(initial=0) > 6e4:
                    qtype, why = BF16_T, "would overflow F16"

                n_params = int(arr.size)
                if (shape_fix and arr.ndim > 1 and n_params >= base.REARRANGE_THRESHOLD
                        and (n_params / 256).is_integer()
                        and not (arr.shape[-1] / 256).is_integer()):
                    writer.add_array(f"comfy.gguf.orig_shape.{name}",
                                     tuple(int(d) for d in arr.shape))
                    arr = arr.reshape(n_params // 256, 256)

                try:
                    out = gguf.quants.quantize(arr, qtype)
                except (AttributeError, gguf.QuantError, ValueError) as e:
                    qtype, why = (F16_T if np.abs(arr).max(initial=0) <= 6e4 else F32_T), \
                        f"fell back: {e}"
                    out = gguf.quants.quantize(arr, qtype)

                writer.add_tensor(name, out, raw_dtype=qtype)
                written += 1
                tally[qtype.name] = tally.get(qtype.name, 0) + 1
                if why and not why.startswith("fell back"):
                    tag = re.sub(r"\[.*?\]", "[..]", why)
                    reasons[tag] = reasons.get(tag, 0) + 1

                if PROGRESS == "every tensor":
                    print(f"  [{written:>4}/{len(todo)}] {name:{width}} {src_dtype:>7} -> "
                          f"{qtype.name:<5} {list(SRC_SHAPES[key]['shape'])}"
                          + (f"  ({why})" if why else ""))
                elif bar:
                    bar.update(1)
                    bar.set_postfix_str(name[-28:], refresh=False)
                elif why.startswith("fell back"):
                    print(f"  {name}: {why}")
                del arr, out

    if bar:
        bar.close()

    print(f"  {written} tensor(s) written: "
          + ", ".join(f"{k}={v}" for k, v in sorted(tally.items(), key=lambda x: -x[1]))
          + (f", {len(held)} held back for the restore pass" if held else ""))
    if reasons:
        print("  type chosen because: " + "; ".join(f"{k} ({v})" for k, v in reasons.items()))
    print("  writing the file out")
    writer.write_header_to_file(path=str(dst))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file(progress=(PROGRESS != "quiet"))
    writer.close()

    if held:
        from safetensors.torch import save_file
        save_file(held, str(STASH / f"fix_5d_tensors_{ARCH}.safetensors"))

    return dst, sorted(held)

In [ ]:
# @title carry_kv · move the metadata across a rewrite pass

def carry_kv(reader, writer, handled=()):
    """Every pass that rewrites a gguf builds a fresh writer, and anything not copied over is
    gone - comfy.gguf.orig_shape.* (how the loader restores SD1/SDXL shapes) and the source
    stamp in particular. ReaderField.contents() decodes strings and arrays of strings
    properly; reading parts[] by hand turned a string array into its first bytes."""
    handled = set(handled) | {"general.architecture", "general.quantization_version",
                              "general.file_type"}
    carried, lost = 0, []

    for key, field in reader.fields.items():
        if key in handled or key.startswith("GGUF.") or not field.types:
            continue
        try:
            vtype = field.types[0]
            if vtype == gguf.GGUFValueType.ARRAY:
                vals = field.contents() if hasattr(field, "contents") else \
                    [field.parts[i].tolist()[0] for i in field.data]
                sub = field.types[1] if len(field.types) > 1 else None
                try:
                    writer.add_key_value(key, list(vals), vtype, sub_type=sub)
                except TypeError:
                    writer.add_array(key, list(vals))
            elif vtype == gguf.GGUFValueType.STRING:
                writer.add_key_value(key, bytes(field.parts[field.data[-1]]).decode("utf-8"),
                                     vtype)
            else:
                writer.add_key_value(key, field.parts[field.data[-1]].tolist()[0], vtype)
            carried += 1
        except Exception:
            lost.append(key)

    if lost:
        print(f"    [!] couldn't carry across: {lost}")
    return carried, lost

In [ ]:
# @title restore_nd · put the stashed tensors back after quantizing

def restore_nd(path, stash_file):
    from safetensors.torch import load_file

    reader = gguf.GGUFReader(str(path))
    arch = read_field(reader, "general.architecture")
    ftype = gguf.LlamaFileType(read_field(reader, "general.file_type"))
    stash = {k: v.numpy() for k, v in load_file(str(stash_file)).items()}

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    carry_kv(reader, writer)

    for tensor in reader.tensors:
        writer.add_tensor(tensor.name, tensor.data, raw_dtype=tensor.tensor_type)
    for key, data in stash.items():
        writer.add_tensor(key, gguf.quants.quantize(data, gguf.GGMLQuantizationType.F32),
                          raw_dtype=gguf.GGMLQuantizationType.F32)

    tmp = path.with_suffix(".fixed.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()

    del reader
    tmp.replace(path)
    print(f"  restored {len(stash)} into {path.name}")

In [ ]:
# @title restore_from_base · undo what llama-quantize did behind the notebook's back

def restore_from_base(path, base_file):
    """llama-quantize reads the base file and writes its own opinion of it. Two of those
    opinions are walked back here.

    Precision. llama-quantize applies the per-family skip lists in lcpp.patch - or, under a
    stand-in, some other family's - and knows nothing about this checkpoint. Everything the
    base file holds as F32 was put there on purpose by pick_base_type (single rows, raw
    Parameters, protected patterns, layers the source kept F32), so any of it that came back
    quantized or downcast is copied from the base. Same for keys_noquant tensors.

    Dimensions. llama.cpp records ggml_n_dims(), which drops trailing 1s, so a [1, dim]
    Parameter leaves as [dim] and ComfyUI refuses it; a Wan-style rule can also add a 1.
    Only size-1 dimensions are put back."""
    def shape_of(tensor):
        return [int(x) for x in reversed(list(tensor.shape)) if int(x)]

    if base_file is None or not Path(base_file).exists():
        return path

    base_reader = gguf.GGUFReader(str(base_file))
    base = {t.name: t for t in base_reader.tensors}
    reader = gguf.GGUFReader(str(path))

    dims, prot = {}, {}
    for tensor in reader.tensors:
        ref = base.get(tensor.name)
        if ref is None:
            continue
        if ref.tensor_type == F32_T and tensor.tensor_type != F32_T:
            prot[tensor.name] = ref
            continue
        if (NOQUANT_PATTERNS and matches_any(tensor.name, NOQUANT_PATTERNS)
                and tensor.tensor_type != ref.tensor_type):
            prot[tensor.name] = ref
            continue
        want, have = shape_of(ref), shape_of(tensor)
        if want != have and [d for d in want if d != 1] == [d for d in have if d != 1]:
            dims[tensor.name] = want

    if not dims and not prot:
        del reader, base_reader
        return path

    writer = gguf.GGUFWriter(path=None, arch=read_field(reader, "general.architecture"),
                             use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(gguf.LlamaFileType(read_field(reader, "general.file_type")))
    carry_kv(reader, writer)

    for tensor in reader.tensors:
        if tensor.name in prot:
            ref = prot[tensor.name]
            writer.add_tensor(tensor.name, ref.data, raw_dtype=ref.tensor_type)
            continue
        data = tensor.data
        if tensor.name in dims:
            data = data.reshape(tuple(dims[tensor.name][:-1]) + (-1,))
        writer.add_tensor(tensor.name, data, raw_dtype=tensor.tensor_type)

    tmp = Path(path).with_suffix(".fixup.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    del reader, base_reader
    tmp.replace(path)

    if dims:
        print(f"    put back the dimension(s) llama-quantize changed on {len(dims)} "
              f"tensor(s): {sorted(dims)[:4]}")
    if prot:
        print(f"    llama-quantize quantized {len(prot)} tensor(s) the base keeps at full "
              f"precision - copied back: {sorted(prot)[:4]}")
    return path

In [ ]:
# @title quantize_python · quantize here when llama.cpp won't take the architecture

def quantize_python(base_file, qtype):
    """Streams the base file tensor by tensor. Anything the base holds as F32 - which is
    everything pick_base_type protected - stays as it is, as do noquant tensors and anything
    whose rows don't divide into the target's blocks."""
    target = PY_TYPES[qtype]
    out = OUT / f"{base_file.name.rsplit('-', 1)[0]}-{qtype}.gguf"
    started = time.time()
    print(f"  {qtype} from {base_file.name} (in python)", flush=True)

    check_base_is_ours(base_file)

    reader = gguf.GGUFReader(str(base_file))
    arch = read_field(reader, "general.architecture")
    ftype = getattr(gguf.LlamaFileType, f"MOSTLY_{qtype}", gguf.LlamaFileType.ALL_F32)

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    carry_kv(reader, writer)

    kept = converted = fallback = 0
    for tensor in reader.tensors:
        name = tensor.name
        shape = [int(x) for x in reversed(list(tensor.shape)) if int(x)]

        protect = (tensor.tensor_type == F32_T or len(shape) > 4
                   or (NOQUANT_PATTERNS and matches_any(name, NOQUANT_PATTERNS))
                   or tensor.tensor_type not in (F16_T, BF16_T, gguf.GGMLQuantizationType.Q8_0))
        if protect:
            writer.add_tensor(name, tensor.data, raw_dtype=tensor.tensor_type)
            kept += 1
            continue

        try:
            arr = gguf.quants.dequantize(tensor.data, tensor.tensor_type).astype(np.float32)
            arr = arr.reshape(shape)
            writer.add_tensor(name, gguf.quants.quantize(arr, target), raw_dtype=target)
            converted += 1
        except Exception:
            writer.add_tensor(name, tensor.data, raw_dtype=tensor.tensor_type)
            fallback += 1
        finally:
            arr = None

        if PROGRESS == "every tensor":
            print(f"    {name} -> {target.name}")

    writer.write_header_to_file(path=str(out))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file(progress=(PROGRESS != "quiet"))
    writer.close()
    del reader

    before = base_file.stat().st_size / 2**30
    after = out.stat().st_size / 2**30
    print(f"    {converted} quantized, {kept} left alone"
          + (f", {fallback} kept their base type (rows don't divide into {qtype} blocks)"
             if fallback else ""))
    print(f"    {after:.2f} GB, {after / before:.0%} of the base file, "
          f"{time.time() - started:.0f}s")
    return out

In [ ]:
# @title relabel_arch · change the architecture string, keep everything else

def relabel_arch(path, arch):
    """Only the metadata string changes - the tensors are copied across untouched, so this
    is not a reconversion. It does write a second copy first, so you need as much free disk
    as the file is big."""
    reader = gguf.GGUFReader(str(path))
    ftype = gguf.LlamaFileType(read_field(reader, "general.file_type"))

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)

    # comfy.gguf.orig_shape.* and the source stamp both live in here
    carried, lost = carry_kv(reader, writer)

    for tensor in reader.tensors:
        writer.add_tensor(tensor.name, tensor.data, raw_dtype=tensor.tensor_type)

    tmp = path.with_suffix(".relabel.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    del reader

    tmp.replace(path)
    return path, carried

In [ ]:
# @title check_base_is_ours · refuse to cut a quant from the wrong file

def check_base_is_ours(base_file):
    """Every quant inherits whatever the base file holds, so this is the last point where
    a mix-up is still cheap to catch. A base from another checkpoint, or one left half
    written by a run that was interrupted, produces a quant that loads and generates
    noise - which looks like a quantization problem and isn't."""
    ok, note = gguf_is_complete(base_file)
    if not ok:
        raise RuntimeError(
            f"{base_file.name}: {note}. Delete it and convert again rather than "
            f"quantizing a file that was left half written.")

    verdict, detail = stamp_matches(base_file)
    if verdict == "different":
        raise RuntimeError(
            f"{base_file.name} was not made from the checkpoint in 1b - {detail}. "
            f"Delete it, or point 1b at the checkpoint it really came from.")
    if verdict == "unstamped":
        print(f"  [!] {base_file.name} carries no source stamp, so there's no way to "
              f"confirm it came from this checkpoint")
    elif verdict == "match":
        print(f"  base file checks out: {detail}")

In [ ]:
# @title quantize · one llama-quantize pass

def quantize(base_file, qtype):
    if QUANT_ENGINE == "python":
        return quantize_python(base_file, qtype)

    check_base_is_ours(base_file)
    out = OUT / f"{base_file.name.rsplit('-', 1)[0]}-{qtype}.gguf"
    started = time.time()
    print(f"  {qtype} from {base_file.name}", flush=True)

    # llama-quantize refuses a quantized input unless told otherwise:
    # "requantizing from type q8_0 is disabled"
    cmd = [str(QUANTIZE)]
    if REQUANTIZE:
        cmd.append("--allow-requantize")
    cmd += [str(base_file), str(out), qtype]

    proc = subprocess.Popen(cmd,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail.append(line)
        if PROGRESS == "every tensor":
            print("   ", line.rstrip())
        elif "converting to" in line or "quantize time" in line:
            print("   ", line.rstrip())
    if proc.wait() != 0:
        print("".join(tail[-25:]))
        raise RuntimeError(f"llama-quantize failed for {qtype}")

    if MODE != "diffusion":
        before = base_file.stat().st_size / 2**30
        after = out.stat().st_size / 2**30
        print(f"    {after:.2f} GB, {after / before:.0%} of the float file, "
              f"{time.time() - started:.0f}s")
        return out

    # llama.cpp writes ggml_n_dims(), which ignores trailing 1s, and applies its own
    # per-family rules rather than this checkpoint's - both are compared against the base
    # file and put back
    restore_from_base(out, base_file)

    if SRC_FP and stamp_matches(out)[0] != "match":
        # llama.cpp copies the metadata forward, so this only fires if that changed
        print("    [!] the source stamp didn't survive llama-quantize, putting it back")
        relabel_arch(out, read_field(gguf.GGUFReader(str(out)), "general.architecture"))

    before = base_file.stat().st_size / 2**30
    after = out.stat().st_size / 2**30
    print(f"    {after:.2f} GB, {after / before:.0%} of the float file, "
          f"{time.time() - started:.0f}s")
    return out

In [ ]:
BUILT = []

if DO_CONVERT and MODE == "diffusion":
    for stale in STASH.glob("fix_5d_tensors_*.safetensors"):
        stale.unlink()
        print("cleared a stale stash file from an earlier run:", stale.name)

    name = SRC.name if SRC.is_dir() else SRC.stem
    started = time.time()

    # every quant is cut from the base file, so one left over from an earlier run is worth
    # reusing even when 1c didn't ask for the base itself. The exception is a file that has
    # already been through restore_nd - llama-quantize can't read the tensors it put back,
    # so that one has to be converted again before anything can be quantized from it.
    base_file, nd = (REUSE.get("mix") or (already_built("mix") if REUSE_OK else None)), []
    if base_file is not None and QUANTS and has_over_4d(base_file):
        print(f"{base_file.name} already has its over-4D tensors back in it, which "
              f"llama-quantize can't read - converting again")
        base_file = None

    if base_file is None:
        size = sum(f.stat().st_size for f in SRC_FILES) / 2**30
        print(f"converting {SRC.name} ({size:.2f} GB) as {ARCH!r}")
        base_file, nd = convert_diffusion(SRC, str(OUT / f"{name}-{{ftype}}.gguf"))
        BUILT.append(base_file)
        print("wrote", base_file.name)
    else:
        print(f"reusing {base_file.name} as the base, not converting again")

    if QUANTS:
        print(f"\nquantizing to {', '.join(QUANTS)}")

    stand_in = bool(STAND_IN and QUANTS and QUANT_ENGINE == "llama")
    if stand_in:
        relabel_arch(base_file, STAND_IN)

    for qtype in QUANTS:
        BUILT.append(quantize(base_file, qtype))

    if stand_in:
        touched = list(dict.fromkeys(BUILT + [base_file]))
        for f in touched:
            relabel_arch(f, ARCH)
        print(f"  put {ARCH!r} back on {len(touched)} file(s)")

    # last step before the checks: the name the file ships under. Everything up to here
    # ran as QUANT_ARCH because llama-quantize only accepts names lcpp.patch adds; only
    # the metadata string changes, the tensors are copied across untouched.
    if SHIP_ARCH and SHIP_ARCH != ARCH:
        for f in list(dict.fromkeys(BUILT + [base_file])):
            if f.exists():
                relabel_arch(f, SHIP_ARCH)
        print(f"\nshipping as {SHIP_ARCH!r} (converted as {ARCH!r})")

    if nd:
        stash_file = STASH / f"fix_5d_tensors_{ARCH}.safetensors"
        print(f"\nrestoring {len(nd)} tensor(s) over 4 dimensions")
        for f in BUILT:
            restore_nd(f, stash_file)
        stash_file.unlink()
    print(f"\nconversion took {time.time() - started:.0f}s")

    # WANT_MIX goes false when the base was reused, so ask EXPECT instead - otherwise a
    # kept file gets deleted as if it were this run's scratch
    if "mix" not in EXPECT and base_file in BUILT:
        base_file.unlink()
        BUILT.remove(base_file)
        print("dropped the intermediate float file")

elif DO_CONVERT and MODE == "llm":
    conv = LCPP_LLM / "convert_hf_to_gguf.py"
    outtype = "auto" if WANT_MIX else "f16"

    reused = REUSE.get("mix") or (already_built("mix") if REUSE_OK else None)
    if reused is not None:
        print(f"reusing {reused.name} as the base, not converting again")
    else:
        cmd = [sys.executable, str(conv), str(SRC), "--outfile",
               str(OUT / f"{SRC.name}-{{ftype}}.gguf"), "--outtype", outtype]
        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
    BUILT = sorted(OUT.glob("*.gguf"))

    if WANT_MMPROJ and REUSE_OK and any(p.name.startswith("mmproj") for p in BUILT):
        print("mmproj is already there, leaving it")
    elif WANT_MMPROJ:
        r = subprocess.run([sys.executable, str(conv), str(SRC), "--mmproj", "--outfile",
                            str(OUT / f"mmproj-{SRC.name}.gguf")],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("mmproj export failed:\n", (r.stdout + r.stderr)[-2000:])
        else:
            BUILT = sorted(OUT.glob("*.gguf"))

    base_file = max([p for p in BUILT if not p.name.startswith("mmproj")],
                    key=lambda p: p.stat().st_size, default=None)
    for qtype in QUANTS:
        BUILT.append(quantize(base_file, qtype))

else:
    print("check only, nothing converted")

print("\nbuilt")
for p in sorted(OUT.glob("*.gguf")):
    label = p.stem.rsplit("-", 1)[-1] if "-" in p.stem else ""
    print(f"  {p.name:52} {p.stat().st_size / 2**30:>6.2f} GB  {label}")

others = [p for p in sorted(OUT.glob("*")) if p.is_file() and p.suffix != ".gguf"]
if others:
    print("alongside")
    for p in others:
        print(f"  {p.name:52} {p.stat().st_size / 2**30:>6.2f} GB")

## Check what came out

In [ ]:
QUANT_TYPES = {t for t in gguf.GGMLQuantizationType
               if t.name not in ("F32", "F16", "BF16", "F64", "I8", "I16", "I32", "I64")}

FILES = sorted(OUT.rglob("*.gguf"))
MMPROJ = [p for p in FILES if p.name.startswith("mmproj")]
MODELS = [p for p in FILES if p not in MMPROJ]
BASE = max(MODELS, key=lambda p: p.stat().st_size) if MODELS else None

REPORT = {}
_current = None
_kind = "model"

if not FILES:
    print(f"no .gguf under {OUT}")
    loose = [p for p in OUT.rglob("*") if p.is_file()]
    for p in loose[:10]:
        print("  ", p.relative_to(OUT))
print("checking:", ", ".join(p.name for p in FILES) or "nothing")

In [ ]:
# @title log · record one result and print it

def log(level, msg):
    REPORT.setdefault(_current, []).append((level, msg))
    print(f"  [{level}] {msg}")

In [ ]:
# @title read_gguf · tensor names, shapes and types

def read_gguf(path):
    reader = gguf.GGUFReader(str(path))
    tensors = {}
    for t in reader.tensors:
        # gguf stores dimensions in the opposite order to torch
        shape = [int(x) for x in reversed(list(t.shape)) if int(x) != 0]
        tensors[t.name] = {"shape": shape, "type": t.tensor_type, "tensor": t}
    return reader, tensors

In [ ]:
# @title check_files_exist · did everything from 1c actually get made

def check_files_exist():
    names = [p.name.upper() for p in FILES]

    if not FILES:
        log("FAIL", "nothing to check - 1c asked for " + ", ".join(EXPECT))
        return

    for label in EXPECT:
        if label == "mix":
            want_end = ([f"-{BASE_QTYPE.upper()}.GGUF"] if BASE_QTYPE
                        else ["-BF16.GGUF", "-F16.GGUF"])
            found = [n for n in names if any(n.endswith(w) for w in want_end)]
            what = f"mix ({BASE_QTYPE})" if BASE_QTYPE else "mix (BF16 or F16)"
        else:
            found = [n for n in names if n.endswith(f"-{label.upper()}.GGUF")]
            what = label
        log("PASS" if found else "FAIL",
            f"{what}: {found[0].lower()}" if found else f"{what}: no file")

    if WANT_MMPROJ and not MMPROJ:
        log("FAIL", "no mmproj file, so the image or audio side is missing and the model "
                    "will only do text")
    elif WANT_MMPROJ:
        log("PASS", f"mmproj: {MMPROJ[0].name}")

    for what, patterns in COMPANIONS:
        hits = [p.name for pat in patterns for p in OUT.rglob(pat) if p.is_file()]
        log("PASS" if hits else "FAIL",
            f"{what}: {', '.join(sorted(set(hits))[:3])}" if hits else f"{what}: missing")

    if MODE == "diffusion" and not COMPANIONS:
        log("WARN", "no companion files ticked in 1e. A diffusion gguf is only the "
                    "transformer, the VAE and text encoder have to ship with it")

In [ ]:
# @title check_arch · will the runtime accept this architecture string

def check_arch(reader):
    arch = read_field(reader, "general.architecture")
    if not arch:
        log("FAIL", "no general.architecture in the file")
        return

    # "pig" and "cow" make the loader detect from tensor names instead of its whitelist.
    # That only works if the loader's own convert.py recognises the weights.
    if arch in ("pig", "cow"):
        names = list(gg)
        hit = loader_detects(REPO_INFO[LOADER], names) if LOADER in REPO_INFO else None
        if hit:
            log("PASS", f"arch is {arch!r} - {LOADER} detects these weights as {hit!r}")
        else:
            log("FAIL", f"arch is {arch!r}, but {LOADER or 'the loader'}'s detect_arch() "
                        f"doesn't recognise these weights - it will refuse the file")
        return

    if TARGET.startswith("ComfyUI"):
        info = REPO_INFO.get(LOADER) if LOADER else None
        allowed = (info["txt"] if _kind == "textenc" else info["img"]) if info else             (TXT_ARCH if _kind == "textenc" else IMG_ARCH)
        who = LOADER or "the listed loaders"
        if arch in allowed:
            log("PASS", f"arch is {arch!r}, {who} accepts it")
        else:
            others = [r for r, i in REPO_INFO.items() if arch in i["img"]]
            log("FAIL", f"arch is {arch!r}, which {who} rejects" +
                (f" - {others[0]} loads it" if others else ""))
    else:
        log("INFO", f"arch is {arch!r}")

    if MODE == "diffusion" and DETECTED and arch != DETECTED[0]:
        if arch in DETECTED:
            log("WARN", f"the weights match {DETECTED}, those families share detection keys")
        else:
            log("FAIL", f"the file says {arch!r} but the weights match {DETECTED[0]!r}")
    elif MODE == "diffusion" and DERIVED:
        log("INFO", f"{arch!r} was derived, no upstream family matched these weights")

In [ ]:
# @title check_source · same tensors, same shapes as the checkpoint

def check_source(gg):
    # quantization bookkeeping (scales, comfy_quant, input_scale) is meant to be gone
    consumed = set(SRC_AUX) if UNPACK else set()
    kept, dropped = strip_prefix([k for k in SRC_SHAPES if k not in consumed])
    if dropped:
        log("FAIL", f"prefix stripping threw away {len(dropped)} source tensors: {dropped[:4]}")

    def logical(old):
        shape = list(SRC_SHAPES[old]["shape"])
        conf = COMFY_LAYERS.get(old) if UNPACK else None
        if conf and len(shape) == 2:          # packed 4/6-bit layouts store fewer columns
            shape[-1] = int(round(shape[-1] * COMFY_PACK.get(conf.get("format"), 1)))
        return shape
    want = {new: logical(old) for old, new in kept.items()}
    missing = sorted(set(want) - set(gg))
    extra = sorted(set(gg) - set(want))

    big = [k for k in missing if len(want[k]) > 4]
    rest = [k for k in missing if len(want[k]) <= 4]

    if big:
        log("FAIL", f"{len(big)} tensor(s) over 4 dimensions never made it back in: {big[:4]}")
    if rest:
        log("FAIL", f"{len(rest)} source tensor(s) missing: {rest[:8]}")
    if extra:
        log("WARN", f"{len(extra)} tensor(s) with no source counterpart: {extra[:8]}")
    if not missing and not extra:
        log("PASS", f"same tensors as the source ({len(gg)})"
                    + (f", {len(consumed)} quantization tensor(s) folded into their weights"
                       if consumed else ""))

    bad = [(k, want[k], gg[k]["shape"]) for k in sorted(set(want) & set(gg))
           if list(want[k]) != list(gg[k]["shape"])]
    if bad:
        log("FAIL", f"{len(bad)} shape mismatch(es):")
        for k, a, b in bad[:8]:
            print(f"           {k}: source {a} -> gguf {b}")
    else:
        log("PASS", "shapes all match")

In [ ]:
# @title check_dtypes · nothing quantized that should have stayed F32

def check_dtypes(gg):
    dist = {}
    for m in gg.values():
        dist[m["type"].name] = dist.get(m["type"].name, 0) + 1
    log("INFO", "dtypes: " + ", ".join(f"{k}={v}" for k, v in
                                       sorted(dist.items(), key=lambda x: -x[1])))

    # a [1, dim] parameter is a single row, exactly as unsafe to quantize as a 1-D
    # tensor. Testing len(shape) == 1 passes it, which is how cap_pad_token and
    # x_pad_token went out as Q8_0 in a file this check called clean - the shape only
    # collapses to [dim] once llama-quantize has rewritten it, so the base file looked
    # fine and the quant cut from it did not
    flat = [n for n, m in gg.items()
            if int(np.prod(m["shape"][:-1] or [1])) == 1 and m["type"] in QUANT_TYPES]
    if flat:
        log("FAIL", f"{len(flat)} single-row tensor(s) quantized - norms, biases and raw "
                    f"nn.Parameters need F32: {flat[:6]}")
    else:
        log("PASS", "no single-row tensor was quantized")

    wrong = sorted(n for n, m in gg.items()
                   if m["type"] != gguf.GGMLQuantizationType.F32
                   and len(m["shape"]) <= 4 and matches_any(n, F32_PATTERNS))

    # a pattern that matches no tensor at all protects nothing, and without this the
    # check passes for having found nothing to look at - which is how a prefixed,
    # misspelled or `^`-anchored entry read as a substring hides an unprotected weight
    dead = [p for p in F32_PATTERNS if not any(matches_any(n, [p]) for n in gg)]

    if not F32_PATTERNS:
        log("INFO", "this family has no weights needing F32 protection")
    elif wrong:
        log("FAIL", f"{len(wrong)} tensor(s) that must stay F32 were quantized: {wrong[:5]}")
    elif dead and len(dead) == len(F32_PATTERNS):
        log("FAIL", f"none of the {len(dead)} protect pattern(s) match any tensor in this "
                    f"file, so nothing was protected: {dead[:5]}")
    elif dead:
        log("WARN", f"{len(dead)} of {len(F32_PATTERNS)} protect pattern(s) match no tensor "
                    f"here: {dead[:5]}")
        log("PASS", f"the other {len(F32_PATTERNS) - len(dead)} protected tensor(s) are F32")
    else:
        log("PASS", f"all {len(F32_PATTERNS)} protected tensor(s) are F32")

    # everything the base file holds F32 has to be F32 here too
    if BASE_TENSORS is not None and _current != (BASE.name if BASE else None):
        down = [n for n, m in gg.items() if n in BASE_TENSORS
                and BASE_TENSORS[n]["type"] == gguf.GGMLQuantizationType.F32
                and m["type"] != gguf.GGMLQuantizationType.F32]
        if down:
            log("FAIL", f"{len(down)} tensor(s) are F32 in the base file but not here: "
                        f"{down[:5]}")
        else:
            log("PASS", "every tensor the base keeps F32 is F32 here")

In [ ]:
# @title check_tokenizer · vocab, embeddings and chat template line up

def check_tokenizer(reader, gg):
    for key in ("tokenizer.ggml.tokens", "tokenizer.ggml.token_type"):
        if reader.fields.get(key) is None:
            log("FAIL", f"{key} missing, the model won't tokenize")
        else:
            log("PASS", f"{key} present")

    n_vocab = None
    f = reader.fields.get("tokenizer.ggml.tokens")
    if f is not None:
        n_vocab = len(f.data)
        log("INFO", f"vocab is {n_vocab}")

    for name in ("token_embd.weight", "output.weight"):
        if name in gg and n_vocab:
            rows = gg[name]["shape"][0]
            if rows != n_vocab:
                log("FAIL", f"{name} has {rows} rows, tokenizer says {n_vocab}")
            else:
                log("PASS", f"{name} matches the vocab ({rows})")

    if SUBTASK in ("text", "vision", "audio") and reader.fields.get("tokenizer.chat_template") is None:
        log("WARN", "no chat template, the runtime will guess a format")

In [ ]:
# @title check_mmproj · is the image or audio tower actually in there

def check_mmproj(gg):
    vision = any(n.startswith("v.") for n in gg)
    audio = any(n.startswith("a.") for n in gg)
    log("INFO", f"vision={vision} audio={audio}")

    if not (vision or audio):
        log("FAIL", "no v.* or a.* tensors, this projector is empty")
    if SUBTASK == "audio" and not audio:
        log("FAIL", "audio task but no a.* tensors, the audio export failed quietly "
                    "and the model is text-only")
    if SUBTASK == "vision" and not vision:
        log("FAIL", "vision task but no v.* tensors")

In [ ]:
# @title check_vs_base · the quant kept everything the float file had

def check_vs_base(gg, base):
    missing = sorted(set(base) - set(gg))
    extra = sorted(set(gg) - set(base))
    bad = [n for n in sorted(set(gg) & set(base))
           if list(gg[n]["shape"]) != list(base[n]["shape"])]

    if missing:
        log("FAIL", f"{len(missing)} tensor(s) lost in quantization: {missing[:8]}")
    if extra:
        log("WARN", f"{len(extra)} tensor(s) the base doesn't have: {extra[:8]}")
    if bad:
        log("FAIL", f"{len(bad)} tensor(s) changed shape during quantization: {bad[:8]}")
    if not (missing or extra or bad):
        log("PASS", f"same tensors and shapes as {BASE.name}")

In [ ]:
# @title check_nan · dequantize a sample and look for broken numbers

def check_nan(gg):
    names = list(gg)
    if NAN_SAMPLE and len(names) > NAN_SAMPLE:
        names = names[::max(1, len(names) // NAN_SAMPLE)]

    checked = skipped = 0
    bad = []
    for n in names:
        t = gg[n]["tensor"]
        try:
            if t.tensor_type in (gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16):
                arr = t.data.astype(np.float32)
            else:
                arr = gguf.quants.dequantize(t.data, t.tensor_type).astype(np.float32)
        except Exception:
            skipped += 1
            continue
        checked += 1
        if not np.isfinite(arr).all():
            bad.append(n)

    for n in bad:
        log("FAIL", f"NaN or Inf in {n!r}")
    if not bad and checked:
        log("PASS", f"no NaN in {checked} sampled tensors ({skipped} couldn't be read)")

In [ ]:
# @title check_stamp · is this file complete, and from the checkpoint in 1b

def check_stamp(path):
    ok, note = gguf_is_complete(path)
    log("PASS" if ok else "FAIL", f"file is complete: {note}" if ok
        else f"file was left half written: {note}")

    verdict, detail = stamp_matches(path)
    if verdict == "match":
        log("PASS", f"made from {detail}")
    elif verdict == "different":
        log("FAIL", f"not made from the checkpoint in 1b - {detail}")
    elif verdict == "unstamped":
        log("WARN", detail)
    else:
        log("INFO", detail)

In [ ]:
# @title check_magnitude · are these numbers the size a weight should be

def check_magnitude(gg, path, sample=12):
    """The check that catches a dropped scale. Nothing about names or shapes is wrong
    when the packed integers get written out as the weights - every tensor is present,
    every shape matches, every dtype is legal. Only the values give it away: a real
    diffusion weight has a standard deviation around 0.005 to 0.1, and raw int8 codes
    land around 10 to 80. Both files pass every other check in this notebook.
    """
    reader = gguf.GGUFReader(str(path))
    big = sorted((t for t in reader.tensors
                  if t.tensor_type not in (gguf.GGMLQuantizationType.F32,)
                  and len([d for d in t.shape if d > 1]) >= 2),
                 key=lambda t: -int(np.prod([int(d) for d in t.shape if d])))[:sample]

    if not big:
        log("INFO", "no multi-row quantized tensors to measure")
        del reader
        return

    stds, bad = [], []
    for t in big:
        try:
            arr = gguf.quants.dequantize(t.data, t.tensor_type).astype(np.float32)
        except Exception:
            continue
        s = float(arr.reshape(-1).std())
        stds.append((t.name, s))
        if not (SANE_STD[0] <= s <= SANE_STD[1]):
            bad.append((t.name, s))
        del arr

    del reader
    if not stds:
        log("INFO", "nothing could be dequantized to measure")
        return

    med = sorted(s for _, s in stds)[len(stds) // 2]

    # what counts as a normal weight is a property of the model, not a constant. A merge
    # can sit well above the usual 0.02 and be entirely correct. When the checkpoint is
    # on hand it settles the question outright, so only fall back to the fixed band when
    # there is nothing to compare against.
    src_med = None
    if SRC is not None and SRC_SHAPES:
        try:
            kept, _ = strip_prefix(list(SRC_SHAPES))
            tail = {t: o for o, t in kept.items()}
            picks = [tail[n] for n, _ in stds if n in tail][:6]
            if picks:
                with SourceReader() as r:
                    vals = [float(r.get(k).to(torch.float32).std()) for k in picks]
                src_med = sorted(vals)[len(vals) // 2]
        except Exception:
            src_med = None

    if src_med:
        ratio = med / (src_med or 1e-12)
        if 0.5 <= ratio <= 2.0:
            log("PASS", f"weight magnitudes match the checkpoint (median std {med:.4g} "
                        f"here, {src_med:.4g} in the source)")
        else:
            log("FAIL", f"weight magnitudes are {ratio:.2f}x the checkpoint's (median "
                        f"std {med:.4g} here, {src_med:.4g} in the source). A scale "
                        f"applied in the wrong direction, or applied twice, lands here.")
        return

    # nothing legitimate in a diffusion model has a standard deviation in the tens.
    # A value that size is an int8 code that never got multiplied by its scale, and it
    # is a hard failure however few tensors show it - the file cannot produce an image.
    insane = [(n, s) for n, s in stds if s > 5 or s != s]
    if insane:
        log("FAIL", f"{len(insane)}/{len(stds)} sampled weights have std "
                    f"{insane[0][1]:.3g} and up, e.g. {insane[0][0]}. Values this size "
                    f"are packed integer codes, not weights - the quantization scales "
                    f"were never applied and this file will render as noise.")
    elif len(bad) > len(stds) // 2:
        log("FAIL", f"{len(bad)}/{len(stds)} sampled weights sit outside "
                    f"{SANE_STD[0]}..{SANE_STD[1]} (median {med:.4g}): "
                    f"{[n for n, _ in bad[:3]]}. Something rescaled these weights.")
    elif bad:
        log("WARN", f"{len(bad)}/{len(stds)} sampled weights look unusual "
                    f"(median {med:.4g}): {[n for n, _ in bad[:3]]}")
    elif med > 0.08:
        log("WARN", f"no checkpoint to compare against, and the median std is {med:.4g} "
                    f"across {len(stds)} sampled where a diffusion weight is usually "
                    f"nearer 0.02. Often just this model, but a scale applied in the "
                    f"wrong direction lands here too.")
    else:
        log("PASS", f"weight magnitudes are plausible (median std {med:.4g} across "
                    f"{len(stds)} sampled)")

In [ ]:
# @title check_ops_coverage · weights nothing will dequantize at runtime

def role_of(name):
    for role, pat in ROLES:
        if re.search(pat, name, re.I):
            return role
    return "other"


def check_ops_coverage(gg):
    """ComfyUI-GGUF replaces five layer types and no others: Linear, Conv2d, Embedding,
    LayerNorm, GroupNorm. Those know how to dequantize a GGMLTensor on the way through.
    A weight consumed by anything else - an RMSNorm, a raw nn.Parameter, a custom module -
    is read straight off the tensor as numbers, still packed.

    Nothing errors when this happens. The file loads, every name and shape is right, and
    the packed bytes are simply used as the weight. The output keeps local detail and
    loses everything that depends on those weights.

    A Linear or Conv weight is 2-D or 4-D, named .weight, and has more than one row.
    Anything else stored as anything but F32/F16 - BF16 included, the node counts BF16 as
    quantized - is on its own. (Conv1d/Conv3d are not patched either; a 2-D or 4-D name can't
    tell those apart, which is what the ComfyUI check at the end is for.)
    """
    raw = []
    for name, m in gg.items():
        # the node treats BF16 as quantized too - only F32 and F16 are read directly
        if m["type"] in (gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16):
            continue
        shape = m["shape"]
        linear_like = (len(shape) in (2, 4) and name.endswith(".weight")
                       and int(np.prod(shape[:-1] or [1])) > 1)
        if not linear_like:
            raw.append((name, m["type"].name, shape))

    if not raw:
        log("PASS", "every quantized tensor is a Linear or Conv weight, so a GGMLOps "
                    "layer will dequantize it")
        return

    by_role = {}
    for name, ttype, shape in raw:
        by_role.setdefault(role_of(name), []).append(name)
    summary = ", ".join(f"{k} ({len(v)})" for k, v in
                        sorted(by_role.items(), key=lambda kv: -len(kv[1])))
    log("FAIL", f"{len(raw)} quantized tensor(s) that no GGMLOps layer will dequantize - "
                f"they will be used as packed bytes: {summary}")
    for name, ttype, shape in raw[:6]:
        log("INFO", f"  {name}  {ttype}  {shape}")

In [ ]:
# @title check_values · did the numbers survive, not just the names

def check_values(gg, path):
    """Names and shapes matching proves nothing. A weight that was rescaled, transposed,
    or written before its quantization scales were applied has exactly the right name and
    exactly the right shape.

    Rebuilds each source tensor - applying scales when the checkpoint is packed - and
    compares it to the same tensor dequantized back out of the gguf. What to expect when
    conversion was clean:

        F16 / BF16   below 0.001      Q4_K_M   0.02 to 0.08
        Q8_0         0.002 to 0.006   Q4_0     0.04 to 0.12

    Read the per-role table, not the tensor list. One role an order of magnitude above
    the others names the stage of the forward pass that broke.
    """
    from safetensors import safe_open
    import torch

    kept, _ = strip_prefix(list(SRC_SHAPES))
    by_tail = {tail: orig for orig, tail in kept.items() if orig not in SRC_AUX}

    # largest few per role - the big weights carry the model and are the ones a
    # conversion fault shows up in first
    picked, seen = [], {}
    for name in gg:
        if name not in by_tail:
            continue
        src_key = by_tail[name]
        n = int(np.prod(SRC_SHAPES[src_key]["shape"]) or 0)
        seen.setdefault(role_of(name), []).append((n, name, src_key))
    for role, items in seen.items():
        items.sort(reverse=True)
        picked += [(role, nm, sk) for _, nm, sk in items[:DIFF_SAMPLE]]

    if not picked:
        log("INFO", "no tensor in this file matches the checkpoint by name")
        return

    per, worst, errors = {}, [], {}
    with SourceReader() as f:
        for role, name, src_key in picked:
            try:
                src = f.get(src_key).to(torch.float32)
                src = src.numpy().reshape(-1)
                out = gguf.quants.dequantize(gg[name]["tensor"].data,
                                             gg[name]["type"]).astype(np.float32).reshape(-1)
            except Exception as e:
                errors.setdefault(f"{type(e).__name__}: {e}", []).append(name)
                continue

            if out.size < src.size:
                log("FAIL", f"{name} has {out.size} values in the gguf and {src.size} in "
                            f"the checkpoint")
                continue
            out = out[:src.size]

            denom = float(np.sqrt((src ** 2).mean())) or 1e-12
            rel = float(np.sqrt(((src - out) ** 2).mean())) / denom
            norm = float(np.linalg.norm(src) * np.linalg.norm(out)) or 1e-12
            cos = float(src @ out / norm)
            per.setdefault(role, []).append((rel, cos, gg[name]["type"].name))
            worst.append((rel, cos, name, gg[name]["type"].name))
            del src, out

    for msg, names in errors.items():
        log("WARN", f"{len(names)} tensor(s) couldn't be compared - {msg} "
                    f"(e.g. {names[0]})")

    if not per:
        log("WARN", "nothing could be compared, so nothing here says whether the values "
                    "in this file match the checkpoint")
        return

    rows = [(r, c, t) for v in per.values() for r, c, t in v]
    med_all = float(np.median([r for r, _, _ in rows]))

    # Q4_K_M is a mixture - llama-quantize leaves some tensors at Q6_K or Q8_0 and cuts
    # the rest to Q4_K. Comparing every role against one file-wide median makes ordinary
    # Q4_K rounding look like a fault next to a tensor that stayed at Q8_0. A tensor is
    # only unusual next to other tensors carrying the same number of bits.
    by_type = {}
    for r, _, t in rows:
        by_type.setdefault(t, []).append(r)
    # the baseline for a type comes from its lower quartile, not its median. If half the
    # tensors of a type are the broken ones they pull the median up with them and hide
    # behind it - the cleanest quarter is what ordinary rounding at that bit depth costs.
    med_type = {t: float(np.percentile(v, 25)) or 1e-9 for t, v in by_type.items()}

    log("INFO", f"value check: {len(rows)} tensor(s), median relative error "
                f"{med_all:.5f}")
    if len(med_type) > 1:
        log("INFO", "  by type: " + ", ".join(
            f"{t} {med_type[t]:.4f} (n={len(by_type[t])})"
            for t in sorted(med_type, key=lambda k: med_type[k])))

    off = []
    def norm(v):
        return float(np.median([r / med_type[t] for r, _, t in v]))

    for role in sorted(per, key=lambda k: -norm(per[k])):
        v = per[role]
        med = float(np.median([r for r, _, _ in v]))
        ratio = norm(v)
        types = "/".join(sorted({t for _, _, t in v}))
        bad = ratio > 4.0 and med > 0.02
        if bad:
            off.append(role)
        log("INFO", f"  {role:<12} n={len(v):<3} {types:<14} median {med:>9.5f}  "
                    f"{ratio:>5.1f}x its type" + ("   <-- out of line" if bad else ""))

    worst.sort(reverse=True)
    if off:
        log("FAIL", f"{', '.join(off)} {'are' if len(off) > 1 else 'is'} far off other "
                    f"tensors carrying the same number of bits, so this is not "
                    f"quantization loss - those weights changed. Worst: "
                    f"{worst[0][2]} ({worst[0][3]}, rel {worst[0][0]:.4f}, "
                    f"cos {worst[0][1]:.4f})")
    elif med_all > 0.15:
        log("FAIL", f"every role is off by {med_all:.3f} - the whole file drifted from "
                    f"the checkpoint, not one part of it")
    else:
        log("PASS", f"values track the checkpoint across every role, allowing for what "
                    f"each tensor's quant type costs (median {med_all:.5f}, worst "
                    f"{worst[0][0]:.4f} on {worst[0][3]})")

In [ ]:
# @title check_companions · what this file needs wired up to run at all

# caption input width -> which text encoder produced it. Families that share tensor
# names can still take different text encoders, and the width is what tells them apart.
CAP_WIDTH = {
    2560: ("Qwen3-4B / Qwen3-VL-4B", "qwen_3_4b.safetensors or the model's own encoder"),
    2304: ("Gemma2-2B", "gemma_2_2b.safetensors"),
    4096: ("a 4096-wide encoder, T5-XXL or similar", "check the model card"),
    3584: ("Qwen2.5-VL-7B", "qwen_2.5_vl_7b.safetensors"),
}


def check_companions(gg, reader):
    """A diffusion gguf is the transformer and nothing else. Nothing in the file records
    which VAE and text encoder belong with it - but the width of the caption projection
    does pin down the encoder, and a mismatched one is the most common reason a file that
    passes every check still produces nothing usable."""
    if "txtfusion.projector.weight" in gg:
        log("INFO", "Krea 2: CLIPLoader type 'krea2' with the Qwen3-VL-4B text encoder, the "
                    "Qwen-Image VAE (qwen_image_vae.safetensors), ComfyUI v0.25 or newer. "
                    "Turbo wants 8 steps and CFG 1.0")
        return
    cap = next((n for n in gg if n.endswith(("cap_embedder.1.weight",
                                             "cap_embedder.0.weight",
                                             "context_embedder.weight"))), None)
    if cap is None:
        log("INFO", "no caption projection here, so the text encoder can't be inferred")
        return

    width = gg[cap]["shape"][-1]
    name, file = CAP_WIDTH.get(width, (None, None))
    if name:
        log("INFO", f"caption features are {width} wide, which is {name} - the text "
                    f"encoder has to be {file}")
    else:
        log("INFO", f"caption features are {width} wide. Whatever text encoder has that "
                    f"hidden size is the one this needs.")

    arch = read_field(reader, "general.architecture")
    if arch == "lumina2":
        log("INFO", "loaded as 'lumina2': CLIPLoader type must be set to lumina2, and "
                    "the VAE is the 16-channel Flux autoencoder (ae.safetensors)")
    log("INFO", "a Turbo or distilled checkpoint wants CFG 1.0 and few steps - carrying "
                "a base model's CFG and step count across produces saturated, broken "
                "output from a file that is entirely correct")

In [ ]:
# @title check_in_comfyui · load the file the way Unet Loader (GGUF) does, and look

NODES = {}


def gguf_node(repo):
    """The GGUF node's loader, ops and dequantizer, imported without its ComfyUI node
    registration (which needs a running server)."""
    import importlib.util
    if repo in NODES:
        return NODES[repo]
    ensure_comfy()
    path = WORK / "nodes" / repo.replace("/", "__")
    if not path.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GGUF_NODE_REF.strip() or "main",
                        f"https://github.com/{repo}", str(path)], check=True)
    pkg = "ggufnode_" + re.sub(r"\W", "_", repo)
    spec = importlib.util.spec_from_file_location(pkg, str(path / "__init__.py"),
                                                  submodule_search_locations=[str(path)])
    sys.modules[pkg] = importlib.util.module_from_spec(spec)
    mods = {}
    for sub in ("dequant", "ops", "loader"):
        s = importlib.util.spec_from_file_location(f"{pkg}.{sub}", str(path / f"{sub}.py"))
        m = importlib.util.module_from_spec(s)
        sys.modules[f"{pkg}.{sub}"] = m
        s.loader.exec_module(m)
        mods[sub] = m
    NODES[repo] = mods
    return mods


class _Grab:
    def __enter__(self):
        import logging
        self.lines = []
        grab = self

        class H(logging.Handler):
            def emit(self, record):
                grab.lines.append(record.getMessage())
        self.h = H()
        self.root = logging.getLogger()
        self.level = self.root.level
        self.root.addHandler(self.h)
        self.root.setLevel(logging.INFO)
        return self

    def __exit__(self, *a):
        self.root.removeHandler(self.h)
        self.root.setLevel(self.level)


def _keys_from_log(lines, word):
    out = []
    for l in lines:
        m = re.search(rf"{word}:\s*(\[.*\])", l, re.S)
        if m and ("unet" in l or "model" in l or "diffusion" in l):
            try:
                out += list(ast.literal_eval(m.group(1)))
            except Exception:
                out.append(m.group(1)[:200])
    return out


def comfy_load(path):
    """(model, logs) or (None, reason)."""
    node = gguf_node(LOADER)
    import comfy.sd
    with _Grab() as g:
        try:
            sd, extra = node["loader"].gguf_sd_loader(str(path))
        except Exception as e:
            return None, f"{LOADER} refuses the file: {e}"
        try:
            model = comfy.sd.load_diffusion_model_state_dict(
                sd, model_options={"custom_operations": node["ops"].GGMLOps()})
        except Exception as e:
            return None, f"ComfyUI failed building the model: {type(e).__name__}: {e}"
    if model is None:
        return None, ("ComfyUI can't tell what model this is - its key names match no "
                      "family this ComfyUI supports (update COMFYUI_REF, or the keys are in "
                      "a layout ComfyUI doesn't convert)")
    return model, g.lines


def check_weights_in_comfyui(model, lines):
    node = gguf_node(LOADER)
    deq = node["dequant"]
    missing = _keys_from_log(lines, "missing")
    unexpected = _keys_from_log(lines, "unexpected")
    ignored = [l for l in lines if "ignoring 'copy_'" in l]
    if missing:
        log("FAIL", f"ComfyUI found {len(missing)} weight(s) missing - they stay at random "
                    f"init, which is noise: {missing[:6]}")
    else:
        log("PASS", "ComfyUI loaded it with no missing weights")
    if unexpected:
        log("WARN", f"{len(unexpected)} tensor(s) ComfyUI didn't use: {unexpected[:6]}")
    if ignored:
        log("FAIL", f"{len(ignored)} weight(s) couldn't be copied into their module (a type "
                    f"no GGUF layer decodes) and stay uninitialised: {ignored[:2]}")

    dm = model.model.diffusion_model
    kept, _ = strip_prefix([k for k in SRC_SHAPES if k not in SRC_AUX])
    src_of = {v: k for k, v in kept.items()}
    params = [(n, p) for n, p in dm.named_parameters() if n in src_of]
    total = sum(1 for _ in dm.named_parameters())
    if not params:
        log("INFO", f"none of the {total} parameter names match the checkpoint's (ComfyUI "
                    f"renamed them - diffusers layout?), so only the missing-key check applies")
        return
    # every small tensor (norms, modulation, biases - where a wrong type hurts most) plus the
    # largest few of each role
    small = [(n, p) for n, p in params if p.shape.numel() <= 4_000_000]
    by_role = {}
    for n, p in params:
        if p.shape.numel() > 4_000_000:
            by_role.setdefault(role_of(n), []).append((p.shape.numel(), n, p))
    big = [(n, p) for items in by_role.values()
           for _, n, p in sorted(items, key=lambda x: -x[0])[:max(DIFF_SAMPLE, 2)]]

    bad, worst, nonfinite, count = [], (1.0, None), [], 0
    with SourceReader() as r:
        for n, p in small + big:
            try:
                got = deq.dequantize_tensor(p, dtype=torch.float32).reshape(-1)
                ref = r.get(src_of[n]).to(torch.float32).reshape(-1)
            except Exception as e:
                bad.append((n, f"unreadable: {type(e).__name__}: {str(e)[:80]}"))
                continue
            count += 1
            if got.numel() != ref.numel():
                bad.append((n, f"{got.numel()} values, source has {ref.numel()}"))
                continue
            if not torch.isfinite(got).all():
                nonfinite.append(n)
                continue
            if float(ref.norm()) == 0:
                continue
            cos = float(torch.nn.functional.cosine_similarity(got, ref, dim=0))
            if cos < worst[0]:
                worst = (cos, n)
            if cos < 0.9:
                bad.append((n, f"cos {cos:.3f}"))
    if nonfinite:
        log("FAIL", f"{len(nonfinite)} weight(s) hold NaN/Inf after loading: {nonfinite[:4]}")
    if bad:
        log("FAIL", f"{len(bad)} of {count} weights, as ComfyUI holds them after loading, "
                    f"don't match the checkpoint: {bad[:4]}")
    elif count:
        log("PASS", f"{count} of {len(params)} weights read back through {LOADER}'s own "
                    f"dequantizer match the checkpoint (worst cos {worst[0]:.4f}, {worst[1]})")


def probe_forward(model, seed=0):
    """One denoising call on a small random latent. The same inputs go to every file, so
    the outputs can be compared directly."""
    mm = model.model
    dm = mm.diffusion_model
    lf = mm.latent_format
    ch = getattr(lf, "latent_channels", 16)
    dims = getattr(lf, "latent_dimensions", 2)
    # image models sharing a video VAE (Wan 2.1 latents) report 3 latent dimensions but
    # take 4-D input, so both shapes are tried, image first unless the task is video
    shapes = [(1, ch, 16, 16), (1, ch, 2, 16, 16)]
    if dims == 3 and WANT_5D:
        shapes.reverse()
    widths = []
    if hasattr(dm, "txtlayers") and hasattr(dm, "txtdim"):
        widths.append(dm.txtlayers * dm.txtdim)
    for attr in ("context_in_dim", "context_dim", "cap_feat_dim", "text_dim", "txt_dim"):
        v = getattr(dm, attr, None)
        if isinstance(v, int):
            widths.append(v)
    for n, p in dm.named_parameters():
        if p.ndim == 2 and re.search(r"(txt_in|context_embedder|cap_embedder|caption_proj|"
                                     r"text_embedding|y_embedder|txt_norm|text_proj|"
                                     r"context_proj|txtmlp)", n):
            widths.append(int(p.shape[-1]))
    widths += [4096, 2560, 3584, 2304, 768, 2048, 1024, 5120]
    # pooled vector and guidance inputs some families (Flux, SDXL, HiDream) require
    extras = {}
    pnames = dict(dm.named_parameters())
    for key in ("vector_in.in_layer.weight", "label_emb.0.0.weight", "pooled_text_proj.weight",
                "y_embedder.mlp.0.weight", "p_embedder.pooled_embedder.linear_1.weight"):
        if key in pnames:
            extras["y"] = torch.randn(1, int(pnames[key].shape[-1]),
                                      generator=torch.Generator().manual_seed(seed + 2))
            break
    if any(n.startswith("guidance_in.") for n in pnames):
        extras["guidance"] = torch.tensor([3.5])

    tried = []
    for shape in shapes:
        x = torch.randn(shape, generator=torch.Generator().manual_seed(seed))
        for w in dict.fromkeys(widths):
            ctx = torch.randn(1, 8, w, generator=torch.Generator().manual_seed(seed + 1))
            for call in ("apply_model", "direct"):
                try:
                    with torch.no_grad():
                        if call == "apply_model":
                            out = mm.apply_model(x, torch.tensor([0.6]), c_crossattn=ctx,
                                                 **extras)
                        else:
                            out = dm(x, torch.tensor([600.0]), ctx, **extras)
                    return out.float(), f"latent {list(shape)}, context width {w}"
                except Exception as e:
                    tried.append(f"{w}/{call}: {type(e).__name__}")
    return None, "; ".join(tried[:4])


FORWARD_REF = {}


def check_in_comfyui(path):
    if COMFYUI_CHECK == "off" or MODE != "diffusion" or not SRC_SHAPES:
        return
    try:
        model, lines = comfy_load(path)
    except Exception as e:
        log("WARN", f"couldn't run the ComfyUI check: {type(e).__name__}: {e}")
        return
    if model is None:
        log("FAIL", lines)
        return
    log("PASS", f"loads in ComfyUI through {LOADER} as {type(model.model).__name__}")
    check_weights_in_comfyui(model, lines)

    if COMFYUI_CHECK == "weights + forward pass":
        out, how = probe_forward(model)
        if out is None:
            log("INFO", f"no generic forward call fits this family ({how}) - weights checked only")
        elif not torch.isfinite(out).all():
            log("FAIL", "the forward pass produces NaN/Inf - this renders as a black or noise image")
        elif path == BASE:
            FORWARD_REF["out"] = out
            log("PASS", f"forward pass runs ({how}), output std {float(out.std()):.4g}")
        elif "out" in FORWARD_REF and FORWARD_REF["out"].shape == out.shape:
            ref = FORWARD_REF["out"]
            cos = float(torch.nn.functional.cosine_similarity(out.flatten(), ref.flatten(), 0))
            rel = float((out - ref).norm() / (ref.norm() + 1e-12))
            msg = f"one denoising step vs {BASE.name}: cos {cos:.4f}, rel {rel:.3f}"
            if cos >= 0.98:
                log("PASS", msg)
            elif cos >= 0.9:
                log("WARN", msg + " - visible quality loss likely")
            else:
                log("FAIL", msg + " - this far off the base, sampling compounds it into noise")
    del model
    import gc; gc.collect()

In [ ]:
if not DO_CHECK:
    print("convert only, skipping the checks")
else:
    _current = "files in the folder"
    REPORT.setdefault(_current, [])
    print("=== files in the folder ===")
    check_files_exist()

    if SRC_BITS and SRC_BITS < 16:
        log("WARN", f"the source was {main_dtype}, only {SRC_BITS} bits per weight, so "
                    f"everything here carries a second rounding step")

    BASE_TENSORS = None
    if COMPARE_QUANTS_TO_BASE and BASE is not None:
        _, BASE_TENSORS = read_gguf(BASE)

    # the base first, so the forward-pass reference exists before the quants are compared
    for path in sorted(FILES, key=lambda p: (p != BASE, p.name)):
        _current = path.name
        _kind = "mmproj" if path in MMPROJ else "model"
        REPORT.setdefault(_current, [])
        print(f"\n=== {path.name}  ({path.stat().st_size / 2**30:.2f} GB) ===")

        reader, gg = read_gguf(path)
        if not gg:
            log("FAIL", "no tensors in this file")
            continue

        check_stamp(path)
        check_arch(reader)
        log("INFO", f"file_type {read_field(reader, 'general.file_type')}, {len(gg)} tensors")

        # llama.cpp renames tensors on the way in, so only the diffusion path matches by name
        if SRC_SHAPES and COMPARE_TO_SOURCE and MODE == "diffusion" and _kind == "model":
            check_source(gg)

        check_dtypes(gg)

        if _kind == "mmproj":
            check_mmproj(gg)
        elif MODE == "llm":
            check_tokenizer(reader, gg)

        if BASE_TENSORS is not None and path != BASE and _kind == "model":
            check_vs_base(gg, BASE_TENSORS)

        if NAN_SAMPLE is not None:
            check_nan(gg)

        if MODE == "diffusion" and _kind == "model":
            check_magnitude(gg, path)
            check_ops_coverage(gg)
            check_companions(gg, reader)
            if DIFF_SAMPLE and COMPARE_TO_SOURCE and SRC is not None and SRC_SHAPES:
                check_values(gg, path)
            check_in_comfyui(path)

        del reader, gg

In [ ]:
# @title Verdict

fails = sum(1 for m in REPORT.values() for lvl, _ in m if lvl == "FAIL")
warns = sum(1 for m in REPORT.values() for lvl, _ in m if lvl == "WARN")
OK = fails == 0 and not (STRICT and warns)

print("=" * 62)
for name, msgs in REPORT.items():
    f = sum(1 for lvl, _ in msgs if lvl == "FAIL")
    w = sum(1 for lvl, _ in msgs if lvl == "WARN")
    print(f"  {'FAIL' if f else 'WARN' if w else 'ok  '}  {name:48} {f} fail  {w} warn")
print("=" * 62)
print(f"\n{fails} fail, {warns} warn  ->  {'good to ship' if OK else 'do not ship'}")

if not OK:
    said = " ".join(m for msgs in REPORT.values() for _, m in msgs)
    steps = []
    if "packed integer codes" in said:
        steps.append("The scales were never applied. Re-run 1f and read the `direction:` "
                     "line in the already-quantized cell - if it says neither direction "
                     "works, the scales aren't in this checkpoint and you need the float "
                     "weights instead.")
    if "no GGMLOps layer will dequantize" in said:
        steps.append("Add the listed names to KEEP_IN_F32 in 1e and convert again. Those "
                     "weights have to ship as F32 because nothing dequantizes them at "
                     "runtime.")
    if "out of line" in said or "far off the rest" in said:
        steps.append("One part of the network drifted while the rest converted cleanly, "
                     "so this is a conversion fault, not quantization loss. Set 1c to "
                     "build only the float base, run the checks on that alone, and see "
                     "whether the same role is still off - that separates a fault in "
                     "convert_diffusion from one in the quantize step.")
    if "missing - they stay at random" in said or "can't tell what model" in said:
        steps.append("ComfyUI doesn't recognise every weight. Update ComfyUI (COMFYUI_REF), and "
                     "convert from a checkpoint with ComfyUI key names (the Comfy-Org repack) "
                     "rather than a diffusers folder.")
    if "refuses the file" in said or "rejects" in said or "doesn't recognise these" in said:
        steps.append("The GGUF node in LOAD_WITH won't open this architecture name. Pick the "
                     "fork that knows the family in 1d (molbal/ComfyUI-GGUF for Krea 2) and "
                     "install that same node in ComfyUI.")
    if "don't match the checkpoint" in said:
        steps.append("Weights changed between the checkpoint and what ComfyUI holds. If the "
                     "source is ComfyUI-quantized, check IF_ALREADY_QUANTIZED is 'unpack'; "
                     "otherwise build only the base in 1c and check it alone.")
    if "sampling compounds it" in said:
        steps.append("The quant drifts too far from the base. Use a higher-bit target, or the "
                     "stand-in K-quants instead of python Q4_0/Q4_1.")
    if "must stay F32" in said or "single-row tensor" in said:
        steps.append("Weights that needed F32 were quantized. Check what the architecture "
                     "cell derived, and add anything missing to KEEP_IN_F32 in 1e.")
    if not steps:
        steps.append("Read the FAIL lines above in order - the first one is usually the "
                     "cause and the rest are consequences.")

    print("\nwhat to do next")
    for i, s in enumerate(steps, 1):
        print(f"  {i}. {s}")
    print("\n  In every case, the fastest way to halve the problem is to build the float")
    print("  base on its own in 1c and load that. If the float file is good, the fault is")
    print("  in quantization. If it shows the same thing, the fault is in conversion and")
    print("  nothing downstream of it is worth debugging yet.")

## Report and upload

In [ ]:
if SAVE_REPORT and REPORT:
    lines = [
        "# GGUF conversion report", "",
        f"- task: `{TASK}` ({MODE} / {SUBTASK})",
        f"- source: `{SRC_LABEL}`",
        f"- source id: `{SRC_FP.get('hash', 'not stamped')}` ({SRC_FP.get('method', 'off')})",
        f"- architecture: `{ARCH}`" + (" (derived)" if DERIVED else "")
        + (f", quantized under stand-in `{STAND_IN}`" if STAND_IN else ""),
        f"- detected: {DETECTED or 'no known family'} via {ARCH_SOURCE}",
        f"- kept in F32: {len(F32_PATTERNS)} pattern(s)"
        + (", plus the tensors the source stores F32" if SOURCE_HAS_F32_INTENT and KEEP_SOURCE_F32 else ""),
        f"- load with: `{LOADER}` (architecture string `{SHIP_ARCH}`)",
        f"- built: {', '.join(EXPECT)}",
        f"- result: **{'good to ship' if OK else 'do not ship'}** ({fails} fail, {warns} warn)",
        "",
    ]
    for name, msgs in REPORT.items():
        if not msgs:
            continue
        lines += [f"## {name}", ""]
        lines += [f"- **{lvl}** {msg}" if lvl in ("FAIL", "WARN") else f"- {lvl} {msg}"
                  for lvl, msg in msgs]
        lines.append("")

    report_path = OUT / "verification_report.md"
    report_path.write_text("\n".join(lines))
    print("wrote", report_path)

if UPLOAD_REPO.strip():
    if not OK:
        print("not uploading, the checks failed. Fix them or upload by hand.")
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOK or None)
        api.create_repo(UPLOAD_REPO.strip(), exist_ok=True)
        api.upload_folder(folder_path=str(OUT), repo_id=UPLOAD_REPO.strip())
        print("uploaded to", UPLOAD_REPO.strip())